## 1. Carregamento da base tratada

A base utilizada nesta etapa é o arquivo gerado no tratamento dos dados.

In [2]:
import pandas as pd

caminho = "../staging/dados_tratados.csv"

df_kpis = pd.read_csv(
    caminho,
    sep=";",
    encoding="utf-8-sig"
)

print("Dimensões da base:", df_kpis.shape)
print("Quantidade de cenários:", df_kpis["Cenario"].nunique())
print("Quantidade de anos:", df_kpis["Ano"].nunique())
print("Quantidade de contas:", df_kpis["Conta"].nunique())

display(df_kpis.head())

Dimensões da base: (958800, 4)
Quantidade de cenários: 1200
Quantidade de anos: 12
Quantidade de contas: 69


,Cenario,Ano,Conta,Valor_numerico
0,Total Cen_00001,Ano 1,BAL - Amortização - Intangível,-5.047124e+09
1,Total Cen_00001,Ano 1,BAL - Amortização Acumulada,-3.011128e+07
2,Total Cen_00001,Ano 1,BAL - At Fiscal Diferido,-1.039693e+08
3,Total Cen_00001,Ano 1,BAL - Ativo Circulante,1.795721e+09
4,Total Cen_00001,Ano 1,BAL - Capital Social,-3.096189e+08


## 2. Consulta de um cenário

Nesta etapa, vamos selecionar um cenário e organizar suas contas por ano.

In [3]:
cenario_selecionado = "Total Cen_00001"

consulta = (
    df_kpis[df_kpis["Cenario"] == cenario_selecionado]
    .pivot_table(
        index="Conta",
        columns="Ano",
        values="Valor_numerico",
        aggfunc="first"
    )
)

ordem_anos = [f"Ano {i}" for i in range(1, 13)]

consulta = consulta.reindex(columns=ordem_anos)

print("Cenário selecionado:", cenario_selecionado)

display(consulta)

Cenário selecionado: Total Cen_00001


Ano,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
Conta,,,,,,,,,,,,
BAL - Amortização - Intangível,-5.047124e+09,-5.710884e+09,-6.417036e+09,-7.165162e+09,-7.936959e+09,-8.723575e+09,-9.525109e+09,-1.034131e+10,-1.116491e+10,-1.200028e+10,-1.284384e+10,-2.990000e-07
BAL - Amortização Acumulada,-3.011128e+07,-4.054538e+07,-5.334893e+07,-6.756197e+07,-8.255313e+07,-1.006275e+08,-1.215114e+08,-1.448155e+08,-1.806699e+08,-2.234075e+08,-2.746443e+08,NaN
BAL - At Fiscal Diferido,-1.039693e+08,1.636680e+08,4.649431e+08,6.293188e+08,6.229204e+08,8.631238e+08,1.201250e+09,1.573346e+09,1.749407e+09,1.688779e+09,1.859161e+09,7.500000e-08
BAL - Ativo Circulante,1.795721e+09,1.567979e+09,2.292584e+09,1.655697e+09,8.393786e+08,7.907685e+08,9.407401e+08,1.779399e+09,2.208646e+09,2.461672e+09,3.209590e+09,3.877167e+08
BAL - Capital Social,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,-3.096189e+08,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
FLU - Receita,5.396831e+09,5.872798e+09,6.123728e+09,6.443489e+09,6.812088e+09,7.237617e+09,7.710044e+09,8.242633e+09,8.742053e+09,9.313009e+09,9.918856e+09,1.345478e+10
FLU - Resultado Financeiro,-7.304044e+08,-5.457156e+08,-4.887332e+08,-1.580896e+09,-1.525871e+09,-1.112855e+09,-1.034734e+09,-3.344773e+08,-2.160440e+08,-1.373751e+08,-8.781890e+07,2.205148e+07
FLU - Saldo Final,1.205265e+09,1.022427e+09,1.748421e+09,1.117390e+09,3.073864e+08,2.661750e+08,4.276930e+08,1.369964e+09,1.923333e+09,2.325378e+09,3.245810e+09,4.181949e+08


## 3. Separação das demonstrações

A consulta do cenário contém contas do Balanço Patrimonial (BAL), da Demonstração do Resultado (DRE) e do Fluxo de Caixa (FLU).

Nesta etapa, vamos separar essas três demonstrações para facilitar a organização e as análises posteriores.

In [4]:
bal = consulta[consulta.index.str.startswith("BAL -")].copy()
dre = consulta[consulta.index.str.startswith("DRE -")].copy()
flu = consulta[consulta.index.str.startswith("FLU -")].copy()

print("Quantidade de contas no BAL:", len(bal))
print("Quantidade de contas na DRE:", len(dre))
print("Quantidade de contas no FLU:", len(flu))

Quantidade de contas no BAL: 43
Quantidade de contas na DRE: 14
Quantidade de contas no FLU: 12


## 4. Conferência das contas

Antes de organizar as demonstrações, vamos conferir quais contas existem em cada uma delas.

In [5]:
print("Contas do Balanço Patrimonial:")
display(bal.index.to_frame(name="Conta").reset_index(drop=True))

print("Contas da DRE:")
display(dre.index.to_frame(name="Conta").reset_index(drop=True))

print("Contas do Fluxo de Caixa:")
display(flu.index.to_frame(name="Conta").reset_index(drop=True))

Contas do Balanço Patrimonial:


,Conta
0,BAL - Amortização - Intangível
1,BAL - Amortização Acumulada
2,BAL - At Fiscal Diferido
3,BAL - Ativo Circulante
4,BAL - Capital Social
5,BAL - Contas a Pagar - Parte Relacionada
6,BAL - Contas a Receber - Clientes
7,BAL - Contas a Receber - Partes Relacionadas
8,BAL - Contas a Receber - SWAP
9,BAL - Créditos Tributários


Contas da DRE:


,Conta
0,DRE - Custos
1,DRE - Depreciação e Amortização
2,DRE - Despesas Financeiras
3,DRE - EBITDA
4,DRE - Imposto de Renda e Contribuição Social
5,DRE - Outros Resultados Operacionais
6,DRE - Receita
7,DRE - Receitas Financeiras
8,DRE - Resultado Antes do Imposto de Renda
9,DRE - Resultado Financeiro


Contas do Fluxo de Caixa:


,Conta
0,FLU - Custos
1,FLU - Despesas Financeiras
2,FLU - Distribuição para Acionista
3,FLU - Entradas
4,FLU - Geração de Caixa
5,FLU - Imposto de Renda e Contribuição Social
6,FLU - Investimentos
7,FLU - Receita
8,FLU - Resultado Financeiro
9,FLU - Saldo Final


## 4. Estrutura das contas por ano

As contas do Balanço Patrimonial não aparecem da mesma forma em todos os anos.

Vamos verificar a quantidade de contas disponíveis em cada demonstração para cada ano do cenário selecionado.

In [6]:
estrutura = pd.DataFrame({
    "BAL": bal.notna().sum(),
    "DRE": dre.notna().sum(),
    "FLU": flu.notna().sum()
})

display(estrutura)

,BAL,DRE,FLU
Ano,,,
Ano 1,42,14,12
Ano 2,42,14,12
Ano 3,43,14,12
Ano 4,43,14,12
Ano 5,43,14,12
Ano 6,43,14,12
Ano 7,43,14,12
Ano 8,43,14,12
Ano 9,43,14,12


## 5. Variação das contas do Balanço Patrimonial

O Balanço Patrimonial possui contas que aparecem ou deixam de aparecer em determinados anos.
Aqui estamos contando, para cada conta do BAL, em quantos dos 12 anos ela possui um valor.

In [7]:
contas_bal_por_ano = bal.notna().sum(axis=1)

contas_variaveis = contas_bal_por_ano[
    contas_bal_por_ano != 12
].sort_values()

display(contas_variaveis.to_frame("Anos com valor"))

,Anos com valor
Conta,
BAL - Dividendos Antecipados,9
BAL - Amortização Acumulada,11
BAL - Contas a Pagar - Parte Relacionada,11
BAL - Capital Social,11
BAL - Contas a Receber - Partes Relacionadas,11
BAL - Contas a Receber - SWAP,11
BAL - Créditos Tributários,11
BAL - Diferido,11
BAL - Emprést,11


## 6. Identificando os anos sem valor

Verificando em quais anos as contas variáveis do Balanço Patrimonial não possuem informação.

In [8]:
contas_variaveis = contas_variaveis.index

anos_sem_valor = bal.loc[contas_variaveis].isna()

display(anos_sem_valor)

Ano,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
Conta,,,,,,,,,,,,
BAL - Dividendos Antecipados,True,True,False,False,False,False,False,False,False,False,False,True
BAL - Amortização Acumulada,False,False,False,False,False,False,False,False,False,False,False,True
BAL - Contas a Pagar - Parte Relacionada,False,False,False,False,False,False,False,False,False,False,False,True
BAL - Capital Social,False,False,False,False,False,False,False,False,False,False,False,True
BAL - Contas a Receber - Partes Relacionadas,False,False,False,False,False,False,False,False,False,False,False,True
BAL - Contas a Receber - SWAP,False,False,False,False,False,False,False,False,False,False,False,True
BAL - Créditos Tributários,False,False,False,False,False,False,False,False,False,False,False,True
BAL - Diferido,False,False,False,False,False,False,False,False,False,False,False,True
BAL - Emprést,False,False,False,False,False,False,False,False,False,False,False,True


## 7. Contas disponíveis nas demonstrações

Vamos listar as contas de cada demonstração para definir a organização que será utilizada na análise.

In [9]:
contas_bal = bal.index.tolist()
contas_dre = dre.index.tolist()
contas_flu = flu.index.tolist()

print("BAL:", contas_bal)
print("DRE:", contas_dre)
print("FLU:", contas_flu)

BAL: ['BAL - Amortização - Intangível', 'BAL - Amortização Acumulada', 'BAL - At Fiscal Diferido', 'BAL - Ativo Circulante', 'BAL - Capital Social', 'BAL - Contas a Pagar - Parte Relacionada', 'BAL - Contas a Receber - Clientes', 'BAL - Contas a Receber - Partes Relacionadas', 'BAL - Contas a Receber - SWAP', 'BAL - Créditos Tributários', 'BAL - Depreciação Acumulada', 'BAL - Diferido', 'BAL - Disponível', 'BAL - Dividendos Antecipados', 'BAL - Emprést', 'BAL - Empréstimos', 'BAL - Encargos Sociais e Trabalhistas', 'BAL - Estoques Diversos', 'BAL - Exigível a Longo Prazo', 'BAL - Fornecedores', 'BAL - Impostos', 'BAL - Investimentos - Imobilizado', 'BAL - Investimentos - Intangível', 'BAL - Obrigações com o Poder Concedente', 'BAL - Outorga da Concessão', 'BAL - Outros Créditos', 'BAL - Outros Créditos LP', 'BAL - Outros Débitos', 'BAL - Outros deb', 'BAL - Passivo Circulante', 'BAL - Patrimônio Líquido', 'BAL - Permanente', 'BAL - Prov para Contingências', 'BAL - Provisão Manutenção',

## 8. Organização da DRE

Vamos organizar as contas da DRE em uma ordem mais próxima da apresentação contábil, mantendo apenas as contas existentes na base.

In [10]:
ordem_dre = [
    "DRE - Receita",
    "DRE - Tributos",
    "DRE - Custos",
    "DRE - Depreciação e Amortização",
    "DRE - Resultado Operacional",
    "DRE - Outros Resultados Operacionais",
    "DRE - Resultado Financeiro",
    "DRE - Resultado Antes do Imposto de Renda",
    "DRE - Imposto de Renda e Contribuição Social",
    "DRE - Resultado Líquido",
    "DRE - Resultado Líquido após Equivalência",
    "DRE - Receitas Financeiras",
    "DRE - Despesas Financeiras",
    "DRE - EBITDA"
]

dre_organizada = dre.reindex(ordem_dre)

display(dre_organizada)

Ano,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
Conta,,,,,,,,,,,,
DRE - Receita,6.722729e+09,6.114937e+09,6.251283e+09,6.506532e+09,6.844711e+09,7.270773e+09,7.720096e+09,8.248988e+09,8.756416e+09,9.334636e+09,9.930604e+09,1.222254e+10
DRE - Tributos,-4.714642e+08,-5.038188e+08,-5.297025e+08,-5.573618e+08,-5.892456e+08,-6.260539e+08,-6.669188e+08,-7.129878e+08,-7.561876e+08,-8.055753e+08,-8.579810e+08,-1.056766e+09
DRE - Custos,-1.361165e+09,-1.333291e+09,-1.232953e+09,-1.236163e+09,-1.127881e+09,-1.140222e+09,-1.269192e+09,-1.342629e+09,-1.412146e+09,-1.277249e+09,-1.051993e+09,-1.094296e+09
DRE - Depreciação e Amortização,-6.035715e+08,-7.282685e+08,-7.787410e+08,-8.139598e+08,-8.293566e+08,-8.473875e+08,-8.705542e+08,-8.894801e+08,-9.044198e+08,-9.326670e+08,-9.559346e+08,-1.060166e+09
DRE - Resultado Operacional,4.286529e+09,3.549559e+09,3.709887e+09,3.899047e+09,4.298228e+09,4.657110e+09,4.913431e+09,5.303891e+09,5.683662e+09,6.319144e+09,7.064696e+09,9.011310e+09
DRE - Outros Resultados Operacionais,3.908440e+07,4.258004e+07,4.726440e+07,4.585229e+07,4.587993e+07,4.727958e+07,4.863268e+07,5.009538e+07,5.167129e+07,5.320341e+07,5.477030e+07,1.473659e+09
DRE - Resultado Financeiro,-6.023477e+08,-5.372516e+08,-5.143071e+08,-4.071543e+08,-3.144898e+08,-1.998613e+08,-1.565845e+08,-1.311405e+08,-4.660607e+07,8.035777e+07,1.857299e+08,2.967618e+08
DRE - Resultado Antes do Imposto de Renda,3.723265e+09,3.054887e+09,3.242845e+09,3.537745e+09,4.029618e+09,4.504528e+09,4.805479e+09,5.222846e+09,5.688727e+09,6.452706e+09,7.305196e+09,1.078173e+10
DRE - Imposto de Renda e Contribuição Social,-1.280022e+09,-1.054053e+09,-1.119699e+09,-1.219618e+09,-1.387012e+09,-1.549159e+09,-1.652172e+09,-1.794810e+09,-1.953967e+09,-2.214465e+09,-2.505081e+09,-3.687911e+09


## 9. Organização do Balanço Patrimonial

Vamos separar as contas do Balanço Patrimonial em Ativo, Passivo e Patrimônio Líquido, mantendo os nomes e valores originais da base.

In [11]:
contas_ativo = [
    conta for conta in contas_bal
    if any(palavra in conta for palavra in [
        "Ativo",
        "Disponível",
        "Contas a Receber",
        "Créditos",
        "Estoques",
        "Investimentos",
        "Outorga",
        "Diferido",
        "Amortização",
        "Depreciação",
        "Realizável",
        "Permanente"
    ])
]

contas_passivo = [
    conta for conta in contas_bal
    if any(palavra in conta for palavra in [
        "Passivo",
        "Fornecedores",
        "Emprést",
        "Tributos",
        "Impostos",
        "Encargos",
        "Obrigações",
        "Provisão",
        "Outros Débitos",
        "Outros deb",
        "Contas a Pagar",
        "Exigível"
    ])
]

contas_pl = [
    conta for conta in contas_bal
    if any(palavra in conta for palavra in [
        "Patrimônio Líquido",
        "Capital Social",
        "Reservas",
        "Resultado",
        "Dividendos"
    ])
]

print("Ativo:")
display(pd.DataFrame({"Conta": contas_ativo}))

print("Passivo:")
display(pd.DataFrame({"Conta": contas_passivo}))

print("Patrimônio Líquido:")
display(pd.DataFrame({"Conta": contas_pl}))

Ativo:


,Conta
0,BAL - Amortização - Intangível
1,BAL - Amortização Acumulada
2,BAL - At Fiscal Diferido
3,BAL - Ativo Circulante
4,BAL - Contas a Receber - Clientes
5,BAL - Contas a Receber - Partes Relacionadas
6,BAL - Contas a Receber - SWAP
7,BAL - Créditos Tributários
8,BAL - Depreciação Acumulada
9,BAL - Diferido


Passivo:


,Conta
0,BAL - Contas a Pagar - Parte Relacionada
1,BAL - Emprést
2,BAL - Empréstimos
3,BAL - Encargos Sociais e Trabalhistas
4,BAL - Exigível a Longo Prazo
5,BAL - Fornecedores
6,BAL - Impostos
7,BAL - Obrigações com o Poder Concedente
8,BAL - Outros Débitos
9,BAL - Outros deb


Patrimônio Líquido:


,Conta
0,BAL - Capital Social
1,BAL - Dividendos Antecipados
2,BAL - Patrimônio Líquido
3,BAL - Reservas Legais
4,BAL - Reservas de Capital
5,BAL - Resultado Acumulado
6,BAL - Resultado do Período


## 10. Organização do Balanço Patrimonial

Agora vamos definir a ordem das contas do BAL para apresentar o Balanço Patrimonial de forma organizada, mantendo os valores originais da base.

In [16]:
ordem_bal = [
    # Ativo
    "BAL - Ativo Circulante",
    "BAL - Disponível",
    "BAL - Contas a Receber - Clientes",
    "BAL - Contas a Receber - Partes Relacionadas",
    "BAL - Contas a Receber - SWAP",
    "BAL - Estoques Diversos",
    "BAL - Créditos Tributários",
    "BAL - Outros Créditos",
    "BAL - Realizável a Longo Prazo",
    "BAL - Outros Créditos LP",
    "BAL - At Fiscal Diferido",
    "BAL - Investimentos - Imobilizado",
    "BAL - Investimentos - Intangível",
    "BAL - Outorga da Concessão",
    "BAL - Diferido",
    "BAL - Amortização - Intangível",
    "BAL - Amortização Acumulada",
    "BAL - Depreciação Acumulada",
    "BAL - Permanente",
    "BAL - Total do Ativo",

    # Passivo
    "BAL - Passivo Circulante",
    "BAL - Fornecedores",
    "BAL - Contas a Pagar - Parte Relacionada",
    "BAL - Emprést",
    "BAL - Empréstimos",
    "BAL - Tributos a pagar",
    "BAL - Impostos",
    "BAL - Encargos Sociais e Trabalhistas",
    "BAL - Obrigações com o Poder Concedente",
    "BAL - Provisão Manutenção",
    "BAL - Prov para Contingências",
    "BAL - Outros Débitos",
    "BAL - Outros deb",
    "BAL - Exigível a Longo Prazo",
    "BAL - Total do Passivo",

    # Patrimônio Líquido
    "BAL - Patrimônio Líquido",
    "BAL - Capital Social",
    "BAL - Reservas Legais",
    "BAL - Reservas de Capital",
    "BAL - Reserva de Retenção de Lucros",
    "BAL - Resultado Acumulado",
    "BAL - Resultado do Período",
    "BAL - Dividendos Antecipados"
]

bal_organizado = bal.reindex(ordem_bal)

print("Quantidade de contas no BAL:", len(bal_organizado))

display(bal_organizado)

Quantidade de contas no BAL: 43


Ano,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
Conta,,,,,,,,,,,,
BAL - Ativo Circulante,1.795721e+09,1.567979e+09,2.292584e+09,1.655697e+09,8.393786e+08,7.907685e+08,9.407401e+08,1.779399e+09,2.208646e+09,2.461672e+09,3.209590e+09,3.877167e+08
BAL - Disponível,1.205265e+09,1.022427e+09,1.748421e+09,1.117390e+09,3.073864e+08,2.661750e+08,4.276930e+08,1.369964e+09,1.923333e+09,2.325378e+09,3.245810e+09,4.181949e+08
BAL - Contas a Receber - Clientes,4.538710e+08,4.055679e+08,4.055679e+08,4.055679e+08,4.055679e+08,4.055679e+08,4.055679e+08,4.055679e+08,4.055679e+08,4.055679e+08,4.055679e+08,-2.393000e-06
BAL - Contas a Receber - Partes Relacionadas,5.846615e+05,5.846615e+05,5.846615e+05,5.846615e+05,5.846615e+05,5.846615e+05,5.846615e+05,5.846615e+05,5.846615e+05,5.846615e+05,5.846615e+05,NaN
BAL - Contas a Receber - SWAP,9.568042e+07,1.625707e+08,2.248569e+08,2.826332e+08,3.398336e+08,3.959129e+08,4.478066e+08,4.075952e+08,3.468343e+08,2.611347e+08,1.518983e+08,NaN
BAL - Estoques Diversos,8.662079e+04,8.662079e+04,8.662079e+04,8.662079e+04,8.662079e+04,8.662079e+04,8.662079e+04,8.662079e+04,8.662079e+04,8.662079e+04,8.662079e+04,NaN
BAL - Créditos Tributários,1.846453e+06,1.846453e+06,1.846453e+06,1.846453e+06,1.846453e+06,1.846453e+06,1.846453e+06,1.846453e+06,1.846453e+06,1.846453e+06,1.846453e+06,NaN
BAL - Outros Créditos,4.023325e+07,-2.325881e+07,-8.693319e+07,-1.505659e+08,-2.140806e+08,-2.775586e+08,-3.409987e+08,-4.043998e+08,-4.677606e+08,-5.310801e+08,-5.943570e+08,4.600000e-08
BAL - Realizável a Longo Prazo,6.390239e+08,9.066612e+08,1.207936e+09,1.372312e+09,1.365914e+09,1.606117e+09,1.944243e+09,2.316339e+09,2.492400e+09,2.431772e+09,2.602154e+09,7.500000e-08


## 11. Organização do Fluxo de Caixa

Agora vamos organizar as contas do Fluxo de Caixa (FLU) na ordem da demonstração, mantendo os valores originais da base.

In [13]:
ordem_flu = [
    "FLU - Entradas",
    "FLU - Receita",
    "FLU - Tributos",
    "FLU - Custos",
    "FLU - Resultado Financeiro",
    "FLU - Despesas Financeiras",
    "FLU - Imposto de Renda e Contribuição Social",
    "FLU - Geração de Caixa",
    "FLU - Investimentos",
    "FLU - Distribuição para Acionista",
    "FLU - Saldo Inicial",
    "FLU - Saldo Final"
]

flu_organizado = flu.reindex(ordem_flu)

display(flu_organizado)

Ano,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
Conta,,,,,,,,,,,,
FLU - Entradas,9.162700e+07,1.515790e+08,1.097470e+08,1.868873e+08,1.179543e+08,3.177291e+07,2.756451e+07,4.429096e+07,1.418705e+08,1.991762e+08,2.264838e+08,3.161307e+08
FLU - Receita,5.396831e+09,5.872798e+09,6.123728e+09,6.443489e+09,6.812088e+09,7.237617e+09,7.710044e+09,8.242633e+09,8.742053e+09,9.313009e+09,9.918856e+09,1.345478e+10
FLU - Tributos,-4.732576e+08,-5.050062e+08,-5.339775e+08,-5.647615e+08,-5.939120e+08,-6.271554e+08,-6.676729e+08,-7.140642e+08,-7.604794e+08,-8.114043e+08,-8.639336e+08,-1.064908e+09
FLU - Custos,-7.616372e+08,-7.857755e+08,-7.826596e+08,-1.283996e+09,-1.662593e+09,-9.458276e+08,-8.614856e+08,-8.997262e+08,-1.540350e+09,-2.008454e+09,-1.126438e+09,-1.029556e+09
FLU - Resultado Financeiro,-7.304044e+08,-5.457156e+08,-4.887332e+08,-1.580896e+09,-1.525871e+09,-1.112855e+09,-1.034734e+09,-3.344773e+08,-2.160440e+08,-1.373751e+08,-8.781890e+07,2.205148e+07
FLU - Despesas Financeiras,-8.220314e+08,-6.972945e+08,-5.984802e+08,-1.767784e+09,-1.643826e+09,-1.144628e+09,-1.062298e+09,-3.787683e+08,-3.579146e+08,-3.365513e+08,-3.143027e+08,-2.940792e+08
FLU - Imposto de Renda e Contribuição Social,-9.344728e+08,-1.333357e+09,-1.374441e+09,-1.337788e+09,-1.334741e+09,-1.742432e+09,-1.942004e+09,-2.117176e+09,-2.078752e+09,-2.101016e+09,-2.621085e+09,-3.840487e+09
FLU - Geração de Caixa,4.306498e+08,-1.828375e+08,7.259936e+08,-6.310307e+08,-8.100039e+08,-4.121136e+07,1.615180e+08,9.422711e+08,5.533691e+08,4.020450e+08,9.204318e+08,-2.827615e+09
FLU - Investimentos,-5.340012e+08,-3.320350e+08,-1.481845e+08,-8.644133e+07,-4.467833e+07,-8.998893e+07,-5.248761e+07,-3.433239e+07,-1.412965e+08,-5.410245e+07,-4.267875e+07,-7.674906e+06


## 12. Validação final das demonstrações

Antes de calcular os indicadores financeiros, vamos conferir se as demonstrações estão estruturadas corretamente.

Nesta etapa, vamos verificar se BAL, DRE e FLU possuem os anos esperados e se não existem contas duplicadas dentro do cenário selecionado.

In [18]:
print("Validação das demonstrações")

print("\nBAL")
print("Contas:", len(bal_organizado))
print("Anos:", bal_organizado.shape[1])
print("Contas duplicadas:", bal_organizado.index.duplicated().sum())

print("\nDRE")
print("Contas:", len(dre_organizada))
print("Anos:", dre_organizada.shape[1])
print("Contas duplicadas:", dre_organizada.index.duplicated().sum())

print("\nFLU")
print("Contas:", len(flu_organizado))
print("Anos:", flu_organizado.shape[1])
print("Contas duplicadas:", flu_organizado.index.duplicated().sum())

print("\nAnos disponíveis:")
print(list(bal_organizado.columns))

Validação das demonstrações

BAL
Contas: 43
Anos: 12
Contas duplicadas: 0

DRE
Contas: 14
Anos: 12
Contas duplicadas: 0

FLU
Contas: 12
Anos: 12
Contas duplicadas: 0

Anos disponíveis:
['Ano 1', 'Ano 2', 'Ano 3', 'Ano 4', 'Ano 5', 'Ano 6', 'Ano 7', 'Ano 8', 'Ano 9', 'Ano 10', 'Ano 11', 'Ano 12']


## 13. Conferência das contas do Balanço Patrimonial

Vamos comparar as contas existentes no BAL original com a lista utilizada para organizar a demonstração.

O objetivo é identificar se alguma conta ficou de fora durante a organização.

In [17]:
contas_bal_originais = set(bal.index)
contas_bal_organizadas = set(bal_organizado.index)

faltantes = contas_bal_originais - contas_bal_organizadas

print("Contas no BAL original:", len(contas_bal_originais))
print("Contas no BAL organizado:", len(contas_bal_organizadas))

print("\nConta(s) que ficaram de fora:")
for conta in sorted(faltantes):
    print("-", conta)

Contas no BAL original: 43
Contas no BAL organizado: 43

Conta(s) que ficaram de fora:


## 12.1. Validação do Balanço Patrimonial

Agora vamos verificar algumas relações contábeis do Balanço Patrimonial.

O objetivo é identificar se os valores fornecidos pela base são coerentes entre si antes de utilizá-los no cálculo dos indicadores financeiros.

In [20]:
ativo = bal_organizado.loc["BAL - Total do Ativo"]
passivo = bal_organizado.loc["BAL - Total do Passivo"]

diferenca_bal = ativo + passivo

print("Validação do Balanço Patrimonial")
print("\nDiferença entre Total do Ativo e Total do Passivo:")

display(diferenca_bal.to_frame("Diferença"))

Validação do Balanço Patrimonial

Diferença entre Total do Ativo e Total do Passivo:


,Diferença
Ano,
Ano 1,0.030001
Ano 2,0.020000
Ano 3,0.020000
Ano 4,0.019999
Ano 5,0.020000
Ano 6,0.020000
Ano 7,0.020000
Ano 8,0.030000
Ano 9,0.030000


## 12.2. Validação da composição do Ativo

Vamos verificar se o Total do Ativo é compatível com a soma do Ativo Circulante, do Realizável a Longo Prazo e do Permanente.

Pequenas diferenças podem ocorrer devido ao arredondamento dos valores fornecidos pela base.

In [21]:
ativo_circulante = bal_organizado.loc["BAL - Ativo Circulante"]
realizavel_lp = bal_organizado.loc["BAL - Realizável a Longo Prazo"]
permanente = bal_organizado.loc["BAL - Permanente"]
total_ativo = bal_organizado.loc["BAL - Total do Ativo"]

diferenca_ativo = (
    ativo_circulante
    + realizavel_lp
    + permanente
    - total_ativo
)

print("Validação da composição do Ativo")
print("\nDiferença entre a composição calculada e o Total do Ativo:")

display(diferenca_ativo.to_frame("Diferença"))

Validação da composição do Ativo

Diferença entre a composição calculada e o Total do Ativo:


,Diferença
Ano,
Ano 1,0.000000e+00
Ano 2,0.000000e+00
Ano 3,9.998322e-03
Ano 4,1.000023e-02
Ano 5,1.000118e-02
Ano 6,-9.536743e-07
Ano 7,9.999275e-03
Ano 8,-1.000023e-02
Ano 9,0.000000e+00


## 12.3. Validação do Patrimônio Líquido

Vamos verificar se o Patrimônio Líquido apresentado na base é compatível com a soma das suas principais contas componentes.

In [22]:
capital_social = bal_organizado.loc["BAL - Capital Social"]
reservas_legais = bal_organizado.loc["BAL - Reservas Legais"]
reservas_capital = bal_organizado.loc["BAL - Reservas de Capital"]
reserva_retencao = bal_organizado.loc["BAL - Reserva de Retenção de Lucros"]
resultado_acumulado = bal_organizado.loc["BAL - Resultado Acumulado"]
resultado_periodo = bal_organizado.loc["BAL - Resultado do Período"]
dividendos_antecipados = bal_organizado.loc["BAL - Dividendos Antecipados"]

pl_calculado = (
    capital_social
    + reservas_legais
    + reservas_capital
    + reserva_retencao
    + resultado_acumulado
    + resultado_periodo
    + dividendos_antecipados
)

pl_informado = bal_organizado.loc["BAL - Patrimônio Líquido"]

diferenca_pl = pl_calculado - pl_informado

print("Validação do Patrimônio Líquido")
display(diferenca_pl.to_frame("Diferença"))

Validação do Patrimônio Líquido


,Diferença
Ano,
Ano 1,NaN
Ano 2,NaN
Ano 3,9.999752e-03
Ano 4,0.000000e+00
Ano 5,1.000023e-02
Ano 6,-2.384186e-07
Ano 7,9.999752e-03
Ano 8,9.999752e-03
Ano 9,4.768372e-07


## 12.4. Verificação das contas do Patrimônio Líquido

Antes de continuar as validações, vamos verificar quais contas do Patrimônio Líquido estão disponíveis em cada ano.

Isso ajuda a diferenciar contas realmente zeradas de contas que não fazem parte da estrutura do ano.

In [23]:
contas_pl = [
    "BAL - Capital Social",
    "BAL - Reservas Legais",
    "BAL - Reservas de Capital",
    "BAL - Reserva de Retenção de Lucros",
    "BAL - Resultado Acumulado",
    "BAL - Resultado do Período",
    "BAL - Dividendos Antecipados"
]

estrutura_pl = bal_organizado.loc[contas_pl].notna()

print("Contas do Patrimônio Líquido disponíveis por ano:")
display(estrutura_pl)

Contas do Patrimônio Líquido disponíveis por ano:


Ano,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
Conta,,,,,,,,,,,,
BAL - Capital Social,True,True,True,True,True,True,True,True,True,True,True,False
BAL - Reservas Legais,True,True,True,True,True,True,True,True,True,True,True,False
BAL - Reservas de Capital,True,True,True,True,True,True,True,True,True,True,True,False
BAL - Reserva de Retenção de Lucros,True,True,True,True,True,True,True,True,True,True,True,False
BAL - Resultado Acumulado,True,True,True,True,True,True,True,True,True,True,True,False
BAL - Resultado do Período,True,True,True,True,True,True,True,True,True,True,True,False
BAL - Dividendos Antecipados,False,False,True,True,True,True,True,True,True,True,True,False


## 12.5. Validação do Resultado Operacional

Vamos verificar se o Resultado Operacional informado na DRE é compatível com a soma das contas que o compõem.

In [24]:
receita = dre_organizada.loc["DRE - Receita"]
tributos = dre_organizada.loc["DRE - Tributos"]
custos = dre_organizada.loc["DRE - Custos"]
da = dre_organizada.loc["DRE - Depreciação e Amortização"]

resultado_operacional = dre_organizada.loc["DRE - Resultado Operacional"]

resultado_operacional_calculado = (
    receita
    + tributos
    + custos
    + da
)

diferenca_dre = (
    resultado_operacional_calculado
    - resultado_operacional
)

print("Validação do Resultado Operacional")
display(diferenca_dre.to_frame("Diferença"))

Validação do Resultado Operacional


,Diferença
Ano,
Ano 1,4.768372e-07
Ano 2,-4.768372e-07
Ano 3,-4.768372e-07
Ano 4,-4.768372e-07
Ano 5,-9.536743e-07
Ano 6,0.000000e+00
Ano 7,0.000000e+00
Ano 8,9.536743e-07
Ano 9,9.536743e-07


## 12.6. Validação do Resultado Antes do Imposto de Renda

Vamos verificar se o Resultado Antes do Imposto de Renda é compatível com o Resultado Operacional, os Outros Resultados Operacionais e o Resultado Financeiro.

In [25]:
resultado_operacional = dre_organizada.loc["DRE - Resultado Operacional"]
outros_resultados = dre_organizada.loc["DRE - Outros Resultados Operacionais"]
resultado_financeiro = dre_organizada.loc["DRE - Resultado Financeiro"]

resultado_antes_ir = dre_organizada.loc[
    "DRE - Resultado Antes do Imposto de Renda"
]

resultado_antes_ir_calculado = (
    resultado_operacional
    + outros_resultados
    + resultado_financeiro
)

diferenca_antes_ir = (
    resultado_antes_ir_calculado
    - resultado_antes_ir
)

print("Validação do Resultado Antes do Imposto de Renda")
display(diferenca_antes_ir.to_frame("Diferença"))

Validação do Resultado Antes do Imposto de Renda


,Diferença
Ano,
Ano 1,4.768372e-07
Ano 2,0.000000e+00
Ano 3,9.999752e-03
Ano 4,0.000000e+00
Ano 5,-9.999752e-03
Ano 6,0.000000e+00
Ano 7,0.000000e+00
Ano 8,-1.000023e-02
Ano 9,0.000000e+00


## 12.7. Validação do Resultado Líquido

Vamos verificar se o Resultado Líquido informado na DRE é compatível com o Resultado Antes do Imposto de Renda e o Imposto de Renda e Contribuição Social.

In [26]:
resultado_antes_ir = dre_organizada.loc[
    "DRE - Resultado Antes do Imposto de Renda"
]

ir_cs = dre_organizada.loc[
    "DRE - Imposto de Renda e Contribuição Social"
]

resultado_liquido = dre_organizada.loc[
    "DRE - Resultado Líquido"
]

resultado_liquido_calculado = (
    resultado_antes_ir
    + ir_cs
)

diferenca_resultado_liquido = (
    resultado_liquido_calculado
    - resultado_liquido
)

print("Validação do Resultado Líquido")
display(
    diferenca_resultado_liquido.to_frame("Diferença")
)

Validação do Resultado Líquido


,Diferença
Ano,
Ano 1,0.000000e+00
Ano 2,0.000000e+00
Ano 3,2.384186e-07
Ano 4,0.000000e+00
Ano 5,1.000023e-02
Ano 6,-1.000023e-02
Ano 7,4.768372e-07
Ano 8,4.768372e-07
Ano 9,0.000000e+00


## 12.8. Comparação entre EBITDA e Resultado Operacional

O EBITDA já é informado diretamente pela base.

Nesta etapa, vamos apenas calcular a diferença entre o EBITDA e o Resultado Operacional para entender a relação entre os dois indicadores, sem alterar os valores originais.

In [27]:
ebitda = dre_organizada.loc["DRE - EBITDA"]
resultado_operacional = dre_organizada.loc[
    "DRE - Resultado Operacional"
]

diferenca_ebitda = ebitda - resultado_operacional

print("Diferença entre EBITDA e Resultado Operacional")
display(
    diferenca_ebitda.to_frame("Diferença")
)

Diferença entre EBITDA e Resultado Operacional


,Diferença
Ano,
Ano 1,9.018521e+08
Ano 2,1.081394e+09
Ano 3,1.173403e+09
Ano 4,1.249019e+09
Ano 5,1.155011e+09
Ano 6,1.177654e+09
Ano 7,1.323483e+09
Ano 8,1.389891e+09
Ano 9,1.453255e+09


## 12.9. Validação do Resultado Financeiro

Nesta etapa, vamos verificar se o Resultado Financeiro informado na base é compatível com as Receitas Financeiras e as Despesas Financeiras.

In [28]:
receitas_financeiras = dre_organizada.loc[
    "DRE - Receitas Financeiras"
]

despesas_financeiras = dre_organizada.loc[
    "DRE - Despesas Financeiras"
]

resultado_financeiro = dre_organizada.loc[
    "DRE - Resultado Financeiro"
]

resultado_financeiro_calculado = (
    receitas_financeiras
    + despesas_financeiras
)

diferenca_resultado_financeiro = (
    resultado_financeiro_calculado
    - resultado_financeiro
)

print("Validação do Resultado Financeiro")
display(
    diferenca_resultado_financeiro.to_frame("Diferença")
)

Validação do Resultado Financeiro


,Diferença
Ano,
Ano 1,0.000000e+00
Ano 2,1.192093e-07
Ano 3,-1.000005e-02
Ano 4,-9.999931e-03
Ano 5,5.960464e-08
Ano 6,9.999990e-03
Ano 7,2.980232e-08
Ano 8,0.000000e+00
Ano 9,7.450581e-09


## 12.10. Validação do Resultado Líquido após Equivalência

Vamos comparar o Resultado Líquido com o Resultado Líquido após Equivalência para verificar como esses valores se relacionam na base.

In [29]:
resultado_liquido = dre_organizada.loc[
    "DRE - Resultado Líquido"
]

resultado_liquido_equivalencia = dre_organizada.loc[
    "DRE - Resultado Líquido após Equivalência"
]

diferenca_equivalencia = (
    resultado_liquido_equivalencia
    - resultado_liquido
)

print("Diferença entre Resultado Líquido após Equivalência e Resultado Líquido")

display(
    diferenca_equivalencia.to_frame("Diferença")
)

Diferença entre Resultado Líquido após Equivalência e Resultado Líquido


,Diferença
Ano,
Ano 1,0.0
Ano 2,0.0
Ano 3,0.0
Ano 4,0.0
Ano 5,0.0
Ano 6,0.0
Ano 7,0.0
Ano 8,0.0
Ano 9,0.0


## 12.11. Validação do Saldo Final do FLU

Nesta etapa, vamos verificar se o Saldo Final do FLU é compatível com o Saldo Inicial e as movimentações de caixa apresentadas na demonstração.

In [30]:
saldo_inicial = flu_organizado.loc[
    "FLU - Saldo Inicial"
]

geracao_caixa = flu_organizado.loc[
    "FLU - Geração de Caixa"
]

investimentos = flu_organizado.loc[
    "FLU - Investimentos"
]

distribuicao = flu_organizado.loc[
    "FLU - Distribuição para Acionista"
]

saldo_final = flu_organizado.loc[
    "FLU - Saldo Final"
]

saldo_final_calculado = (
    saldo_inicial
    + geracao_caixa
    + investimentos
    + distribuicao
)

diferenca_saldo_final = (
    saldo_final_calculado
    - saldo_final
)

print("Validação do Saldo Final do FLU")

display(
    diferenca_saldo_final.to_frame("Diferença")
)

Validação do Saldo Final do FLU


,Diferença
Ano,
Ano 1,-2.066409e+09
Ano 2,-2.885781e+09
Ano 3,-2.217923e+09
Ano 4,-2.307078e+09
Ano 5,-2.504975e+09
Ano 6,-2.850558e+09
Ano 7,-3.042630e+09
Ano 8,-3.234919e+09
Ano 9,-3.593059e+09


## 12.11. Estrutura do FLU

Antes de validar os cálculos do fluxo de caixa, vamos visualizar todas as contas do FLU do Cenário 1 ao longo dos 12 anos.

Isso permite identificar como a demonstração está estruturada antes de definir as relações que serão validadas.

In [31]:
id="4xq8sm"
print("FLU do cenário:", cenario_selecionado)

display(flu_organizado)

FLU do cenário: Total Cen_00001


Ano,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
Conta,,,,,,,,,,,,
FLU - Entradas,9.162700e+07,1.515790e+08,1.097470e+08,1.868873e+08,1.179543e+08,3.177291e+07,2.756451e+07,4.429096e+07,1.418705e+08,1.991762e+08,2.264838e+08,3.161307e+08
FLU - Receita,5.396831e+09,5.872798e+09,6.123728e+09,6.443489e+09,6.812088e+09,7.237617e+09,7.710044e+09,8.242633e+09,8.742053e+09,9.313009e+09,9.918856e+09,1.345478e+10
FLU - Tributos,-4.732576e+08,-5.050062e+08,-5.339775e+08,-5.647615e+08,-5.939120e+08,-6.271554e+08,-6.676729e+08,-7.140642e+08,-7.604794e+08,-8.114043e+08,-8.639336e+08,-1.064908e+09
FLU - Custos,-7.616372e+08,-7.857755e+08,-7.826596e+08,-1.283996e+09,-1.662593e+09,-9.458276e+08,-8.614856e+08,-8.997262e+08,-1.540350e+09,-2.008454e+09,-1.126438e+09,-1.029556e+09
FLU - Resultado Financeiro,-7.304044e+08,-5.457156e+08,-4.887332e+08,-1.580896e+09,-1.525871e+09,-1.112855e+09,-1.034734e+09,-3.344773e+08,-2.160440e+08,-1.373751e+08,-8.781890e+07,2.205148e+07
FLU - Despesas Financeiras,-8.220314e+08,-6.972945e+08,-5.984802e+08,-1.767784e+09,-1.643826e+09,-1.144628e+09,-1.062298e+09,-3.787683e+08,-3.579146e+08,-3.365513e+08,-3.143027e+08,-2.940792e+08
FLU - Imposto de Renda e Contribuição Social,-9.344728e+08,-1.333357e+09,-1.374441e+09,-1.337788e+09,-1.334741e+09,-1.742432e+09,-1.942004e+09,-2.117176e+09,-2.078752e+09,-2.101016e+09,-2.621085e+09,-3.840487e+09
FLU - Geração de Caixa,4.306498e+08,-1.828375e+08,7.259936e+08,-6.310307e+08,-8.100039e+08,-4.121136e+07,1.615180e+08,9.422711e+08,5.533691e+08,4.020450e+08,9.204318e+08,-2.827615e+09
FLU - Investimentos,-5.340012e+08,-3.320350e+08,-1.481845e+08,-8.644133e+07,-4.467833e+07,-8.998893e+07,-5.248761e+07,-3.433239e+07,-1.412965e+08,-5.410245e+07,-4.267875e+07,-7.674906e+06


## 12.12. Relação entre Saldo Inicial e Saldo Final

Vamos verificar como o saldo de caixa evolui de um ano para o outro.

Primeiro, vamos comparar o Saldo Inicial de cada ano com o Saldo Final do ano anterior. Essa relação deve ajudar a identificar a lógica utilizada pelo fluxo de caixa.

In [32]:
saldo_inicial = flu_organizado.loc[
    "FLU - Saldo Inicial"
]

saldo_final = flu_organizado.loc[
    "FLU - Saldo Final"
]

comparacao_saldos = pd.DataFrame({
    "Saldo Inicial": saldo_inicial,
    "Saldo Final": saldo_final,
    "Saldo Final do Ano Anterior": saldo_final.shift(1)
})

print("Relação entre os saldos do FLU:")

display(comparacao_saldos)

Relação entre os saldos do FLU:


,Saldo Inicial,Saldo Final,Saldo Final do Ano Anterior
Ano,,,
Ano 1,7.746152e+08,1.205265e+09,NaN
Ano 2,1.205265e+09,1.022427e+09,1.205265e+09
Ano 3,1.022427e+09,1.748421e+09,1.022427e+09
Ano 4,1.748421e+09,1.117390e+09,1.748421e+09
Ano 5,1.117390e+09,3.073864e+08,1.117390e+09
Ano 6,3.073864e+08,2.661750e+08,3.073864e+08
Ano 7,2.661750e+08,4.276930e+08,2.661750e+08
Ano 8,4.276930e+08,1.369964e+09,4.276930e+08
Ano 9,1.369964e+09,1.923333e+09,1.369964e+09


## 12.13. Variação do saldo de caixa

Agora vamos calcular quanto o saldo de caixa aumentou ou diminuiu em cada ano.

Essa variação será comparada com as linhas de movimentação do FLU para identificar como o Saldo Final é formado.

In [33]:
saldo_inicial = flu_organizado.loc[
    "FLU - Saldo Inicial"
]

saldo_final = flu_organizado.loc[
    "FLU - Saldo Final"
]

variacao_saldo = saldo_final - saldo_inicial

print("Variação do saldo de caixa:")

display(
    variacao_saldo.to_frame("Variação do Saldo")
)

Variação do saldo de caixa:


,Variação do Saldo
Ano,
Ano 1,4.306498e+08
Ano 2,-1.828375e+08
Ano 3,7.259936e+08
Ano 4,-6.310307e+08
Ano 5,-8.100039e+08
Ano 6,-4.121136e+07
Ano 7,1.615180e+08
Ano 8,9.422711e+08
Ano 9,5.533691e+08


## 12.14. Validação do Saldo Final do FLU

A análise anterior mostrou que a variação do saldo de caixa é igual à Geração de Caixa.

Portanto, vamos validar se:

Saldo Final = Saldo Inicial + Geração de Caixa

In [34]:
saldo_inicial = flu_organizado.loc[
    "FLU - Saldo Inicial"
]

geracao_caixa = flu_organizado.loc[
    "FLU - Geração de Caixa"
]

saldo_final = flu_organizado.loc[
    "FLU - Saldo Final"
]

saldo_final_calculado = (
    saldo_inicial
    + geracao_caixa
)

diferenca_saldo_final = (
    saldo_final_calculado
    - saldo_final
)

print("Validação do Saldo Final do FLU:")

display(
    diferenca_saldo_final.to_frame("Diferença")
)

Validação do Saldo Final do FLU:


,Diferença
Ano,
Ano 1,-9.999990e-03
Ano 2,0.000000e+00
Ano 3,0.000000e+00
Ano 4,0.000000e+00
Ano 5,9.999990e-03
Ano 6,0.000000e+00
Ano 7,0.000000e+00
Ano 8,0.000000e+00
Ano 9,0.000000e+00


## 13. Índices de Liquidez

Os índices de liquidez mostram a capacidade financeira da empresa de cumprir suas obrigações.

Nesta etapa, vamos calcular:
- Liquidez Corrente (LC)
- Liquidez Seca (LS)
- Liquidez Imediata (LI)
- Liquidez Geral (LG)

Os cálculos serão feitos para os 12 anos do cenário selecionado.

In [35]:
# Contas utilizadas nos índices de liquidez

ativo_circulante = bal_organizado.loc[
    "BAL - Ativo Circulante"
]

disponivel = bal_organizado.loc[
    "BAL - Disponível"
]

estoques = bal_organizado.loc[
    "BAL - Estoques Diversos"
]

realizavel_lp = bal_organizado.loc[
    "BAL - Realizável a Longo Prazo"
]

passivo_circulante = bal_organizado.loc[
    "BAL - Passivo Circulante"
]

exigivel_lp = bal_organizado.loc[
    "BAL - Exigível a Longo Prazo"
]

pc = passivo_circulante.abs()
el = exigivel_lp.abs()

# Índices de liquidez

lc = ativo_circulante / pc

ls = (ativo_circulante - estoques) / pc

li = disponivel / pc

lg = (ativo_circulante + realizavel_lp) / (pc + el)

# Tabela final dos índices

indices_liquidez = pd.DataFrame({
    "Liquidez Corrente (LC)": lc,
    "Liquidez Seca (LS)": ls,
    "Liquidez Imediata (LI)": li,
    "Liquidez Geral (LG)": lg
})

print("Índices de liquidez —", cenario_selecionado)

display(indices_liquidez)

Índices de liquidez — Total Cen_00001


,Liquidez Corrente (LC),Liquidez Seca (LS),Liquidez Imediata (LI),Liquidez Geral (LG)
Ano,,,,
Ano 1,1.661277,1.661197,1.115028,0.282561
Ano 2,1.154874,1.154811,0.753056,0.278254
Ano 3,1.301103,1.301054,0.992276,0.376489
Ano 4,3.324041,3.323867,2.243316,0.376906
Ano 5,0.664478,0.664409,0.243336,0.250630
Ano 6,0.395148,0.395105,0.133008,0.251326
Ano 7,0.379970,0.379935,0.172747,0.288164
Ano 8,0.764404,0.764366,0.588516,0.415432
Ano 9,0.809218,0.809186,0.704683,0.458381


## 14. Índices de Endividamento

Os índices de endividamento mostram como a empresa está financiando seus ativos e qual é a participação do capital de terceiros.

Nesta etapa, vamos calcular:
- Participação de Capital de Terceiros (PCT)
- Composição do Endividamento (CE)
- Imobilização do Patrimônio Líquido (IPL)

In [36]:
# Contas utilizadas

passivo_circulante = bal_organizado.loc[
    "BAL - Passivo Circulante"
].abs()

exigivel_lp = bal_organizado.loc[
    "BAL - Exigível a Longo Prazo"
].abs()

patrimonio_liquido = bal_organizado.loc[
    "BAL - Patrimônio Líquido"
].abs()

permanente = bal_organizado.loc[
    "BAL - Permanente"
]

# Capital de terceiros
capital_terceiros = passivo_circulante + exigivel_lp

# Índices de endividamento

pct = capital_terceiros / patrimonio_liquido

ce = passivo_circulante / capital_terceiros

ipl = permanente / patrimonio_liquido

# Tabela

indices_endividamento = pd.DataFrame({
    "PCT": pct,
    "CE": ce,
    "IPL": ipl
})

print("Índices de endividamento —", cenario_selecionado)

display(indices_endividamento)

Índices de endividamento — Total Cen_00001


,PCT,CE,IPL
Ano,,,
Ano 1,4.277473,0.125446,4.068824
Ano 2,6.085068,0.152663,5.391876
Ano 3,6.137439,0.189511,4.826759
Ano 4,4.982482,0.062000,4.104556
Ano 5,4.902671,0.143564,3.266220
Ano 6,4.793570,0.209836,2.577095
Ano 7,4.650721,0.247296,2.010339
Ano 8,4.142171,0.236112,1.465344
Ano 9,3.850995,0.266129,1.036045


## 15. Validação do EBITDA por engenharia reversa

O EBITDA já é informado na base original.

Para validar esse indicador, vamos comparar o EBITDA informado com diferentes combinações das contas da DRE, buscando identificar a composição utilizada no modelo original.

A validação será feita sem alterar o EBITDA fornecido pela empresa.

In [37]:
# Contas da DRE

receita = dre_organizada.loc["DRE - Receita"]
tributos = dre_organizada.loc["DRE - Tributos"]
custos = dre_organizada.loc["DRE - Custos"]
da = dre_organizada.loc["DRE - Depreciação e Amortização"]

resultado_operacional = dre_organizada.loc[
    "DRE - Resultado Operacional"
]

outros_resultados = dre_organizada.loc[
    "DRE - Outros Resultados Operacionais"
]

ebitda_informado = dre_organizada.loc[
    "DRE - EBITDA"
]

# Possíveis reconstruções

ebitda_1 = resultado_operacional - da

ebitda_2 = receita + tributos + custos

ebitda_3 = resultado_operacional + (-da)

ebitda_4 = (
    resultado_operacional
    + (-da)
    + outros_resultados
)

# Comparação das diferenças

validacao_ebitda = pd.DataFrame({
    "EBITDA informado": ebitda_informado,
    "RO - D&A": ebitda_1,
    "Receita + Tributos + Custos": ebitda_2,
    "RO + (-D&A)": ebitda_3,
    "RO + (-D&A) + Outros Resultados": ebitda_4
})

print("Engenharia reversa do EBITDA —", cenario_selecionado)

display(validacao_ebitda)

Engenharia reversa do EBITDA — Total Cen_00001


,EBITDA informado,RO - D&A,Receita + Tributos + Custos,RO + (-D&A),RO + (-D&A) + Outros Resultados
Ano,,,,,
Ano 1,5.188381e+09,4.890100e+09,4.890100e+09,4.890100e+09,4.929185e+09
Ano 2,4.630952e+09,4.277827e+09,4.277827e+09,4.277827e+09,4.320407e+09
Ano 3,4.883290e+09,4.488628e+09,4.488628e+09,4.488628e+09,4.535893e+09
Ano 4,5.148066e+09,4.713007e+09,4.713007e+09,4.713007e+09,4.758859e+09
Ano 5,5.453240e+09,5.127585e+09,5.127585e+09,5.127585e+09,5.173465e+09
Ano 6,5.834764e+09,5.504497e+09,5.504497e+09,5.504497e+09,5.551777e+09
Ano 7,6.236914e+09,5.783985e+09,5.783985e+09,5.783985e+09,5.832618e+09
Ano 8,6.693782e+09,6.193371e+09,6.193371e+09,6.193371e+09,6.243467e+09
Ano 9,7.136917e+09,6.588082e+09,6.588082e+09,6.588082e+09,6.639753e+09


## 16. Engenharia reversa do EBITDA

As primeiras reconstruções não reproduziram o EBITDA informado na base.

Agora vamos calcular a diferença entre o EBITDA informado e o resultado operacional ajustado, para identificar quais contas da DRE podem explicar essa diferença.

In [38]:
# EBITDA informado
ebitda = dre_organizada.loc[
    "DRE - EBITDA"
]

# Resultado operacional
resultado_operacional = dre_organizada.loc[
    "DRE - Resultado Operacional"
]

# Depreciação e amortização
da = dre_organizada.loc[
    "DRE - Depreciação e Amortização"
]

# Outros resultados operacionais
outros_resultados = dre_organizada.loc[
    "DRE - Outros Resultados Operacionais"
]

# Reconstrução mais próxima encontrada até agora
ebitda_reconstruido = (
    resultado_operacional
    - da
    + outros_resultados
)

# Diferença que ainda precisamos explicar
diferenca = ebitda - ebitda_reconstruido

engenharia_ebitda = pd.DataFrame({
    "EBITDA informado": ebitda,
    "EBITDA reconstruído": ebitda_reconstruido,
    "Diferença": diferenca
})

print("Engenharia reversa do EBITDA —", cenario_selecionado)

display(engenharia_ebitda)

Engenharia reversa do EBITDA — Total Cen_00001


,EBITDA informado,EBITDA reconstruído,Diferença
Ano,,,
Ano 1,5.188381e+09,4.929185e+09,2.591961e+08
Ano 2,4.630952e+09,4.320407e+09,3.105452e+08
Ano 3,4.883290e+09,4.535893e+09,3.473974e+08
Ano 4,5.148066e+09,4.758859e+09,3.892069e+08
Ano 5,5.453240e+09,5.173465e+09,2.797749e+08
Ano 6,5.834764e+09,5.551777e+09,2.829873e+08
Ano 7,6.236914e+09,5.832618e+09,4.042964e+08
Ano 8,6.693782e+09,6.243467e+09,4.503152e+08
Ano 9,7.136917e+09,6.639753e+09,4.971636e+08


## 17. Investigação da composição do EBITDA

A diferença entre o EBITDA informado e o EBITDA reconstruído ainda não foi explicada.

Nesta etapa, vamos comparar essa diferença com todas as contas da DRE para identificar se alguma delas corresponde ao ajuste utilizado no cálculo original.

In [39]:
diferenca_ebitda = (
    dre_organizada.loc["DRE - EBITDA"]
    - (
        dre_organizada.loc["DRE - Resultado Operacional"]
        - dre_organizada.loc["DRE - Depreciação e Amortização"]
        + dre_organizada.loc["DRE - Outros Resultados Operacionais"]
    )
)

# Todas as contas da DRE do cenário
investigacao_ebitda = dre_organizada.copy()

# Adiciona a diferença como uma linha de referência
investigacao_ebitda.loc["AJUSTE NECESSÁRIO PARA EBITDA"] = diferenca_ebitda

print("Investigação da composição do EBITDA —", cenario_selecionado)

display(investigacao_ebitda)

Investigação da composição do EBITDA — Total Cen_00001


Ano,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
Conta,,,,,,,,,,,,
DRE - Receita,6.722729e+09,6.114937e+09,6.251283e+09,6.506532e+09,6.844711e+09,7.270773e+09,7.720096e+09,8.248988e+09,8.756416e+09,9.334636e+09,9.930604e+09,1.222254e+10
DRE - Tributos,-4.714642e+08,-5.038188e+08,-5.297025e+08,-5.573618e+08,-5.892456e+08,-6.260539e+08,-6.669188e+08,-7.129878e+08,-7.561876e+08,-8.055753e+08,-8.579810e+08,-1.056766e+09
DRE - Custos,-1.361165e+09,-1.333291e+09,-1.232953e+09,-1.236163e+09,-1.127881e+09,-1.140222e+09,-1.269192e+09,-1.342629e+09,-1.412146e+09,-1.277249e+09,-1.051993e+09,-1.094296e+09
DRE - Depreciação e Amortização,-6.035715e+08,-7.282685e+08,-7.787410e+08,-8.139598e+08,-8.293566e+08,-8.473875e+08,-8.705542e+08,-8.894801e+08,-9.044198e+08,-9.326670e+08,-9.559346e+08,-1.060166e+09
DRE - Resultado Operacional,4.286529e+09,3.549559e+09,3.709887e+09,3.899047e+09,4.298228e+09,4.657110e+09,4.913431e+09,5.303891e+09,5.683662e+09,6.319144e+09,7.064696e+09,9.011310e+09
DRE - Outros Resultados Operacionais,3.908440e+07,4.258004e+07,4.726440e+07,4.585229e+07,4.587993e+07,4.727958e+07,4.863268e+07,5.009538e+07,5.167129e+07,5.320341e+07,5.477030e+07,1.473659e+09
DRE - Resultado Financeiro,-6.023477e+08,-5.372516e+08,-5.143071e+08,-4.071543e+08,-3.144898e+08,-1.998613e+08,-1.565845e+08,-1.311405e+08,-4.660607e+07,8.035777e+07,1.857299e+08,2.967618e+08
DRE - Resultado Antes do Imposto de Renda,3.723265e+09,3.054887e+09,3.242845e+09,3.537745e+09,4.029618e+09,4.504528e+09,4.805479e+09,5.222846e+09,5.688727e+09,6.452706e+09,7.305196e+09,1.078173e+10
DRE - Imposto de Renda e Contribuição Social,-1.280022e+09,-1.054053e+09,-1.119699e+09,-1.219618e+09,-1.387012e+09,-1.549159e+09,-1.652172e+09,-1.794810e+09,-1.953967e+09,-2.214465e+09,-2.505081e+09,-3.687911e+09


## 19. Margem EBITDA

A Margem EBITDA mostra quanto do faturamento da empresa corresponde ao EBITDA.

O cálculo será feito dividindo o EBITDA pela Receita.

In [40]:
# Contas utilizadas

ebitda = dre_organizada.loc[
    "DRE - EBITDA"
]

receita = dre_organizada.loc[
    "DRE - Receita"
]

# Cálculo da Margem EBITDA

margem_ebitda = ebitda / receita

# Tabela do indicador

margem_ebitda_df = pd.DataFrame({
    "EBITDA": ebitda,
    "Receita": receita,
    "Margem EBITDA": margem_ebitda
})

print("Margem EBITDA —", cenario_selecionado)

display(margem_ebitda_df)

Margem EBITDA — Total Cen_00001


,EBITDA,Receita,Margem EBITDA
Ano,,,
Ano 1,5.188381e+09,6.722729e+09,0.771767
Ano 2,4.630952e+09,6.114937e+09,0.757318
Ano 3,4.883290e+09,6.251283e+09,0.781166
Ano 4,5.148066e+09,6.506532e+09,0.791215
Ano 5,5.453240e+09,6.844711e+09,0.796709
Ano 6,5.834764e+09,7.270773e+09,0.802496
Ano 7,6.236914e+09,7.720096e+09,0.807880
Ano 8,6.693782e+09,8.248988e+09,0.811467
Ano 9,7.136917e+09,8.756416e+09,0.815050


## 20. Verificação dos componentes do EVA

O EVA mede a geração de valor econômico da empresa.

Para calculá-lo, precisamos identificar três componentes:
- NOPAT
- Capital Investido
- WACC

Antes do cálculo, vamos verificar quais informações estão disponíveis na base tratada.

In [41]:
# Procurar contas relacionadas aos componentes do EVA

contas_eva = [
    conta for conta in df_kpis["Conta"].unique()
    if any(
        termo in conta.lower()
        for termo in [
            "nopat",
            "capital investido",
            "wacc",
            "custo de capital"
        ]
    )
]

print("Contas relacionadas ao EVA encontradas na base:")

if contas_eva:
    for conta in contas_eva:
        print("-", conta)
else:
    print("Nenhuma conta encontrada.")

Contas relacionadas ao EVA encontradas na base:
Nenhuma conta encontrada.


## 21. Verificação dos dados necessários para o EVA

A base não possui contas explícitas de NOPAT, Capital Investido ou WACC.

Por isso, vamos verificar se existem contas que permitam derivar esses componentes a partir das demonstrações contábeis.

In [42]:
# Contas que podem ser utilizadas na análise do EVA

contas_eva_base = [
    "DRE - Resultado Operacional",
    "DRE - Resultado Antes do Imposto de Renda",
    "DRE - Imposto de Renda e Contribuição Social",
    "DRE - EBITDA",
    "BAL - Total do Ativo",
    "BAL - Disponível",
    "BAL - Patrimônio Líquido",
    "BAL - Emprést",
    "BAL - Empréstimos",
    "BAL - Exigível a Longo Prazo",
    "BAL - Passivo Circulante"
]

verificacao_eva = pd.DataFrame(
    index=contas_eva_base,
    columns=ordem_anos
)

for conta in contas_eva_base:
    if conta in consulta.index:
        verificacao_eva.loc[conta] = consulta.loc[conta]
    else:
        verificacao_eva.loc[conta] = None

print("Dados disponíveis para análise do EVA —", cenario_selecionado)

display(verificacao_eva)

Dados disponíveis para análise do EVA — Total Cen_00001


,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
DRE - Resultado Operacional,4286528586.65,3549558630.51,3709887225.83,3899047059.26,4298228260.3,4657109696.71,4913430943.64,5303891370.46,5683661942.0,6319144400.73,7064696174.06,9011309885.42
DRE - Resultado Antes do Imposto de Renda,3723265284.69,3054887031.11,3242844525.4,3537745032.17,4029618401.71,4504527975.87,4805479143.81,5222846271.26,5688727159.55,6452705573.8,7305196369.19,10781730602.34
DRE - Imposto de Renda e Contribuição Social,-1280022316.37,-1054053192.92,-1119699015.79,-1219617542.94,-1387011918.91,-1549158733.31,-1652172120.34,-1794809700.54,-1953967477.34,-2214465048.92,-2505080659.59,-3687911169.99
DRE - EBITDA,5188380666.78,4630952327.38,4883290026.3,5148066090.8,5453239715.47,5834764048.96,6236914248.63,6693782013.63,7136916587.56,7653142113.98,8150124768.47,11609874889.190001
BAL - Total do Ativo,10631127075.26,10354991705.75,10812725089.290001,9646282980.629999,8067280208.37,7524099143.88,7212632777.38,7583472267.63,7460182465.36,6817269552.98,6845811932.5,89954344.2
BAL - Disponível,1205264965.52,1022427444.78,1748421038.98,1117390302.73,307386372.0,266175008.15,427692977.25,1369964064.21,1923333134.54,2325378098.86,3245809906.77,418194930.32
BAL - Patrimônio Líquido,-2014435236.96,-1461523233.76,-1514930565.46,-1612421555.27,-1794731721.0,-1989532007.72,-2152696881.56,-2380147199.17,-2663144169.32,-3102772895.5,-3646418677.94,NaN
BAL - Emprést,-5649256635.7,-5649256635.7,-5649256635.7,-5649256635.7,-5649256635.7,-5649256635.7,-5649256635.7,-5644644236.5,-5639889922.12,-5634923652.39,-5629633704.99,NaN
BAL - Empréstimos,-279102364.24,-328383898.3,-369283812.71,838791171.26,2040214126.29,2898691224.85,3757705498.2,4077710571.79,4418330346.11,4781895772.18,5168360855.64,-0.0
BAL - Exigível a Longo Prazo,-7535763775.37,-7535763775.37,-7535763775.37,-7535763775.37,-7535763775.37,-7535763775.37,-7535763775.37,-7531151376.17,-7526397061.79,-7521430792.07,-7516140844.67,NaN


## 22. Investigação da taxa implícita da dívida

A base não apresenta uma taxa de juros de forma explícita.

Como existem despesas financeiras e contas de empréstimos, vamos investigar se é possível identificar uma taxa implícita de custo da dívida.

Essa análise é exploratória e não significa, ainda, que a taxa encontrada seja o WACC.

In [43]:
# Valores financeiros utilizados na investigação

despesas_financeiras = dre_organizada.loc[
    "DRE - Despesas Financeiras"
].abs()

emprestimo = bal_organizado.loc[
    "BAL - Emprést"
].abs()

emprestimos = bal_organizado.loc[
    "BAL - Empréstimos"
].abs()

exigivel_lp = bal_organizado.loc[
    "BAL - Exigível a Longo Prazo"
].abs()

passivo_circulante = bal_organizado.loc[
    "BAL - Passivo Circulante"
].abs()

# Dívida identificada diretamente pelos empréstimos

divida_emprestimos = emprestimo + emprestimos

# Taxa implícita simples

taxa_emprestimos = despesas_financeiras / divida_emprestimos

# Tabela

taxa_divida = pd.DataFrame({
    "Despesas Financeiras": despesas_financeiras,
    "Empréstimo": emprestimo,
    "Empréstimos": emprestimos,
    "Dívida por Empréstimos": divida_emprestimos,
    "Taxa Implícita": taxa_emprestimos
})

print("Investigação da taxa implícita da dívida —", cenario_selecionado)

display(taxa_divida)

Investigação da taxa implícita da dívida — Total Cen_00001


,Despesas Financeiras,Empréstimo,Empréstimos,Dívida por Empréstimos,Taxa Implícita
Ano,,,,,
Ano 1,6.939747e+08,5.649257e+09,2.791024e+08,5.928359e+09,0.117060
Ano 2,6.888306e+08,5.649257e+09,3.283839e+08,5.977641e+09,0.115235
Ano 3,6.240541e+08,5.649257e+09,3.692838e+08,6.018540e+09,0.103689
Ano 4,5.940416e+08,5.649257e+09,8.387912e+08,6.488048e+09,0.091559
Ano 5,4.324440e+08,5.649257e+09,2.040214e+09,7.689471e+09,0.056238
Ano 6,2.316342e+08,5.649257e+09,2.898691e+09,8.547948e+09,0.027098
Ano 7,1.841490e+08,5.649257e+09,3.757705e+09,9.406962e+09,0.019576
Ano 8,1.754314e+08,5.644644e+09,4.077711e+09,9.722355e+09,0.018044
Ano 9,1.884766e+08,5.639890e+09,4.418330e+09,1.005822e+10,0.018739


## 23. Investigação da taxa da dívida pela dívida média

A taxa anterior foi calculada usando o saldo da dívida no final de cada ano.

Agora vamos testar uma segunda aproximação, usando a dívida média entre o ano atual e o ano anterior. Isso ajuda a verificar se as despesas financeiras apresentam uma relação mais consistente com o nível de endividamento.

Essa taxa continua sendo apenas uma estimativa exploratória do custo da dívida e não deve ser tratada como WACC.

In [44]:
# Dívida total identificada pelas contas de empréstimos

divida = (
    bal_organizado.loc["BAL - Emprést"].abs()
    + bal_organizado.loc["BAL - Empréstimos"].abs()
)

# Dívida média entre o ano anterior e o ano atual
divida_media = (divida + divida.shift(1)) / 2

# Taxa implícita usando a dívida média
taxa_divida_media = despesas_financeiras / divida_media

taxa_divida_media_df = pd.DataFrame({
    "Despesas Financeiras": despesas_financeiras,
    "Dívida Final": divida,
    "Dívida Média": divida_media,
    "Taxa Implícita pela Dívida Média": taxa_divida_media
})

print("Taxa implícita pela dívida média —", cenario_selecionado)

display(taxa_divida_media_df)

Taxa implícita pela dívida média — Total Cen_00001


,Despesas Financeiras,Dívida Final,Dívida Média,Taxa Implícita pela Dívida Média
Ano,,,,
Ano 1,6.939747e+08,5.928359e+09,NaN,NaN
Ano 2,6.888306e+08,5.977641e+09,5.953000e+09,0.115712
Ano 3,6.240541e+08,6.018540e+09,5.998090e+09,0.104042
Ano 4,5.940416e+08,6.488048e+09,6.253294e+09,0.094997
Ano 5,4.324440e+08,7.689471e+09,7.088759e+09,0.061004
Ano 6,2.316342e+08,8.547948e+09,8.118709e+09,0.028531
Ano 7,1.841490e+08,9.406962e+09,8.977455e+09,0.020512
Ano 8,1.754314e+08,9.722355e+09,9.564658e+09,0.018342
Ano 9,1.884766e+08,1.005822e+10,9.890288e+09,0.019057


## 24. Verificação dos componentes do EVA

O EVA depende de três informações principais:

- NOPAT
- Capital Investido
- WACC

Já verificamos que a base não apresenta o WACC de forma explícita.

Agora vamos verificar se NOPAT e Capital Investido podem ser obtidos a partir das contas disponíveis, sem criar premissas arbitrárias.

In [45]:
# Contas disponíveis que podem participar da análise do EVA

contas_eva = [
    "DRE - Resultado Operacional",
    "DRE - Imposto de Renda e Contribuição Social",
    "BAL - Ativo Circulante",
    "BAL - Realizável a Longo Prazo",
    "BAL - Permanente",
    "BAL - Total do Ativo",
    "BAL - Passivo Circulante",
    "BAL - Exigível a Longo Prazo",
    "BAL - Patrimônio Líquido"
]

componentes_eva = bal_organizado.reindex(
    [conta for conta in contas_eva if conta in bal_organizado.index]
)

# Adiciona as contas da DRE que existem no demonstrativo
componentes_eva_dre = dre_organizada.reindex(
    [conta for conta in contas_eva if conta in dre_organizada.index]
)

print("Componentes disponíveis para investigação do EVA —", cenario_selecionado)

print("\nContas do Balanço:")
display(componentes_eva)

print("\nContas da DRE:")
display(componentes_eva_dre)

Componentes disponíveis para investigação do EVA — Total Cen_00001

Contas do Balanço:


Ano,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
Conta,,,,,,,,,,,,
BAL - Ativo Circulante,1.795721e+09,1.567979e+09,2.292584e+09,1.655697e+09,8.393786e+08,7.907685e+08,9.407401e+08,1.779399e+09,2.208646e+09,2.461672e+09,3.209590e+09,3.877167e+08
BAL - Realizável a Longo Prazo,6.390239e+08,9.066612e+08,1.207936e+09,1.372312e+09,1.365914e+09,1.606117e+09,1.944243e+09,2.316339e+09,2.492400e+09,2.431772e+09,2.602154e+09,7.500000e-08
BAL - Permanente,8.196382e+09,7.880352e+09,7.312205e+09,6.618274e+09,5.861988e+09,5.127214e+09,4.327649e+09,3.487735e+09,2.759136e+09,1.923826e+09,1.034067e+09,-2.977624e+08
BAL - Total do Ativo,1.063113e+10,1.035499e+10,1.081273e+10,9.646283e+09,8.067280e+09,7.524099e+09,7.212633e+09,7.583472e+09,7.460182e+09,6.817270e+09,6.845812e+09,8.995434e+07
BAL - Passivo Circulante,-1.080928e+09,-1.357705e+09,-1.762031e+09,-4.980976e+08,1.263215e+09,2.001197e+09,2.475828e+09,2.327826e+09,2.729359e+09,3.806934e+09,4.316748e+09,-1.151397e+08
BAL - Exigível a Longo Prazo,-7.535764e+09,-7.535764e+09,-7.535764e+09,-7.535764e+09,-7.535764e+09,-7.535764e+09,-7.535764e+09,-7.531151e+09,-7.526397e+09,-7.521431e+09,-7.516141e+09,NaN
BAL - Patrimônio Líquido,-2.014435e+09,-1.461523e+09,-1.514931e+09,-1.612422e+09,-1.794732e+09,-1.989532e+09,-2.152697e+09,-2.380147e+09,-2.663144e+09,-3.102773e+09,-3.646419e+09,NaN



Contas da DRE:


Ano,Ano 1,Ano 2,Ano 3,Ano 4,Ano 5,Ano 6,Ano 7,Ano 8,Ano 9,Ano 10,Ano 11,Ano 12
Conta,,,,,,,,,,,,
DRE - Resultado Operacional,4.286529e+09,3.549559e+09,3.709887e+09,3.899047e+09,4.298228e+09,4.657110e+09,4.913431e+09,5.303891e+09,5.683662e+09,6.319144e+09,7.064696e+09,9.011310e+09
DRE - Imposto de Renda e Contribuição Social,-1.280022e+09,-1.054053e+09,-1.119699e+09,-1.219618e+09,-1.387012e+09,-1.549159e+09,-1.652172e+09,-1.794810e+09,-1.953967e+09,-2.214465e+09,-2.505081e+09,-3.687911e+09


## 25. Investigação do Capital Investido

O Capital Investido não está apresentado diretamente na base.

Por isso, vamos verificar como os principais componentes do Balanço se relacionam entre si.

Nesta etapa, os cálculos são apenas exploratórios. O objetivo é identificar uma composição que seja consistente com a estrutura da base antes de utilizá-la no EVA.

In [46]:
# Componentes do Balanço utilizados na investigação

ativo_circulante = bal_organizado.loc[
    "BAL - Ativo Circulante"
]

realizavel_lp = bal_organizado.loc[
    "BAL - Realizável a Longo Prazo"
]

permanente = bal_organizado.loc[
    "BAL - Permanente"
]

passivo_circulante = bal_organizado.loc[
    "BAL - Passivo Circulante"
]

exigivel_lp = bal_organizado.loc[
    "BAL - Exigível a Longo Prazo"
]

patrimonio_liquido = bal_organizado.loc[
    "BAL - Patrimônio Líquido"
]

# Ativo operacional / total de ativos
capital_ativo = (
    ativo_circulante
    + realizavel_lp
    + permanente
)

# Capital de terceiros
capital_terceiros = (
    passivo_circulante.abs()
    + exigivel_lp.abs()
)

# Estrutura de capital pela soma dos valores absolutos
capital_total = (
    capital_terceiros
    + patrimonio_liquido.abs()
)

investigacao_capital = pd.DataFrame({
    "Ativo Total": capital_ativo,
    "Capital de Terceiros": capital_terceiros,
    "Patrimônio Líquido": patrimonio_liquido.abs(),
    "Capital Total": capital_total
})

print("Investigação do Capital Investido —", cenario_selecionado)

display(investigacao_capital)

Investigação do Capital Investido — Total Cen_00001


,Ativo Total,Capital de Terceiros,Patrimônio Líquido,Capital Total
Ano,,,,
Ano 1,1.063113e+10,8.616692e+09,2.014435e+09,1.063113e+10
Ano 2,1.035499e+10,8.893468e+09,1.461523e+09,1.035499e+10
Ano 3,1.081273e+10,9.297795e+09,1.514931e+09,1.081273e+10
Ano 4,9.646283e+09,8.033861e+09,1.612422e+09,9.646283e+09
Ano 5,8.067280e+09,8.798979e+09,1.794732e+09,1.059371e+10
Ano 6,7.524099e+09,9.536960e+09,1.989532e+09,1.152649e+10
Ano 7,7.212633e+09,1.001159e+10,2.152697e+09,1.216429e+10
Ano 8,7.583472e+09,9.858978e+09,2.380147e+09,1.223912e+10
Ano 9,7.460182e+09,1.025576e+10,2.663144e+09,1.291890e+10


## 20. Margem Líquida

A Margem Líquida mostra quanto da receita permanece como resultado líquido após os efeitos das demais despesas e impostos.

In [47]:
resultado_liquido = dre_organizada.loc[
    "DRE - Resultado Líquido"
]

receita = dre_organizada.loc[
    "DRE - Receita"
]

margem_liquida = resultado_liquido / receita

margem_liquida_df = pd.DataFrame({
    "Resultado Líquido": resultado_liquido,
    "Receita": receita,
    "Margem Líquida": margem_liquida
})

print("Margem Líquida —", cenario_selecionado)

display(margem_liquida_df)

Margem Líquida — Total Cen_00001


,Resultado Líquido,Receita,Margem Líquida
Ano,,,
Ano 1,2.443243e+09,6.722729e+09,0.363430
Ano 2,2.000834e+09,6.114937e+09,0.327204
Ano 3,2.123146e+09,6.251283e+09,0.339634
Ano 4,2.318127e+09,6.506532e+09,0.356277
Ano 5,2.642606e+09,6.844711e+09,0.386080
Ano 6,2.955369e+09,7.270773e+09,0.406472
Ano 7,3.153307e+09,7.720096e+09,0.408454
Ano 8,3.428037e+09,8.248988e+09,0.415571
Ano 9,3.734760e+09,8.756416e+09,0.426517


## 21. Consolidação dos indicadores financeiros

Nesta etapa, reunimos os indicadores calculados em uma única base.

A tabela final será utilizada posteriormente para comparar os cenários e realizar o ranking.

In [48]:
# Consolidando os indicadores calculados para o cenário selecionado

kpis_cenario = pd.DataFrame({
    "Cenario": cenario_selecionado,
    "Ano": ordem_anos,
    "Receita": receita.values,
    "Custos": custos.values,
    "Resultado Líquido": resultado_liquido.values,
    "EBITDA": ebitda.values,
    "Margem EBITDA": margem_ebitda.values,
    "Margem Líquida": margem_liquida.values,
    "Liquidez Corrente": lc.values,
    "Liquidez Seca": ls.values,
    "Liquidez Imediata": li.values,
    "Liquidez Geral": lg.values,
    "PCT": pct.values,
    "CE": ce.values,
    "IPL": ipl.values
})

print("Tabela consolidada de KPIs —", cenario_selecionado)

display(kpis_cenario)

Tabela consolidada de KPIs — Total Cen_00001


,Cenario,Ano,Receita,Custos,Resultado Líquido,EBITDA,Margem EBITDA,Margem Líquida,Liquidez Corrente,Liquidez Seca,Liquidez Imediata,Liquidez Geral,PCT,CE,IPL
0,Total Cen_00001,Ano 1,6.722729e+09,-1.361165e+09,2.443243e+09,5.188381e+09,0.771767,0.363430,1.661277,1.661197,1.115028,0.282561,4.277473,0.125446,4.068824
1,Total Cen_00001,Ano 2,6.114937e+09,-1.333291e+09,2.000834e+09,4.630952e+09,0.757318,0.327204,1.154874,1.154811,0.753056,0.278254,6.085068,0.152663,5.391876
2,Total Cen_00001,Ano 3,6.251283e+09,-1.232953e+09,2.123146e+09,4.883290e+09,0.781166,0.339634,1.301103,1.301054,0.992276,0.376489,6.137439,0.189511,4.826759
3,Total Cen_00001,Ano 4,6.506532e+09,-1.236163e+09,2.318127e+09,5.148066e+09,0.791215,0.356277,3.324041,3.323867,2.243316,0.376906,4.982482,0.062000,4.104556
4,Total Cen_00001,Ano 5,6.844711e+09,-1.127881e+09,2.642606e+09,5.453240e+09,0.796709,0.386080,0.664478,0.664409,0.243336,0.250630,4.902671,0.143564,3.266220
5,Total Cen_00001,Ano 6,7.270773e+09,-1.140222e+09,2.955369e+09,5.834764e+09,0.802496,0.406472,0.395148,0.395105,0.133008,0.251326,4.793570,0.209836,2.577095
6,Total Cen_00001,Ano 7,7.720096e+09,-1.269192e+09,3.153307e+09,6.236914e+09,0.807880,0.408454,0.379970,0.379935,0.172747,0.288164,4.650721,0.247296,2.010339
7,Total Cen_00001,Ano 8,8.248988e+09,-1.342629e+09,3.428037e+09,6.693782e+09,0.811467,0.415571,0.764404,0.764366,0.588516,0.415432,4.142171,0.236112,1.465344
8,Total Cen_00001,Ano 9,8.756416e+09,-1.412146e+09,3.734760e+09,7.136917e+09,0.815050,0.426517,0.809218,0.809186,0.704683,0.458381,3.850995,0.266129,1.036045
9,Total Cen_00001,Ano 10,9.334636e+09,-1.277249e+09,4.238241e+09,7.653142e+09,0.819865,0.454034,0.646628,0.646606,0.610827,0.431964,3.651045,0.336053,0.620034


## 21. Consolidação dos KPIs

Nesta etapa, os indicadores calculados anteriormente serão aplicados a todos os cenários e anos disponíveis na base.

O resultado será uma tabela consolidada com 1.200 cenários e 12 anos, totalizando 14.400 observações.

In [ ]:
# Consolidação dos indicadores para todos os cenários

registros = []

for cenario in df_kpis["Cenario"].unique():

    dados_cenario = (
        df_kpis[df_kpis["Cenario"] == cenario]
        .pivot_table(
            index="Conta",
            columns="Ano",
            values="Valor_numerico",
            aggfunc="first"
        )
    )

    dados_cenario = dados_cenario.reindex(columns=ordem_anos)

    # DRE
    receita = dados_cenario.loc["DRE - Receita"]
    custos = dados_cenario.loc["DRE - Custos"]
    ebitda = dados_cenario.loc["DRE - EBITDA"]
    resultado_liquido = dados_cenario.loc["DRE - Resultado Líquido"]

    # BP
    ativo_circulante = dados_cenario.loc["BAL - Ativo Circulante"]
    disponivel = dados_cenario.loc["BAL - Disponível"]
    estoques = dados_cenario.loc["BAL - Estoques Diversos"]
    realizavel_lp = dados_cenario.loc["BAL - Realizável a Longo Prazo"]
    passivo_circulante = dados_cenario.loc["BAL - Passivo Circulante"].abs()
    exigivel_lp = dados_cenario.loc["BAL - Exigível a Longo Prazo"].abs()
    patrimonio_liquido = dados_cenario.loc["BAL - Patrimônio Líquido"].abs()
    permanente = dados_cenario.loc["BAL - Permanente"]

    # Indicadores
    margem_ebitda = ebitda / receita
    margem_liquida = resultado_liquido / receita

    lc = ativo_circulante / passivo_circulante
    ls = (ativo_circulante - estoques) / passivo_circulante
    li = disponivel / passivo_circulante
    lg = (ativo_circulante + realizavel_lp) / (
        passivo_circulante + exigivel_lp
    )

    capital_terceiros = passivo_circulante + exigivel_lp

    pct = capital_terceiros / patrimonio_liquido
    ce = passivo_circulante / capital_terceiros
    ipl = permanente / patrimonio_liquido

    for i, ano in enumerate(ordem_anos):

        registros.append({
            "Cenario": cenario,
            "Ano": ano,
            "Receita": receita.iloc[i],
            "Custos": custos.iloc[i],
            "EBITDA": ebitda.iloc[i],
            "Margem EBITDA": margem_ebitda.iloc[i],
            "Resultado Líquido": resultado_liquido.iloc[i],
            "Margem Líquida": margem_liquida.iloc[i],
            "Liquidez Corrente": lc.iloc[i],
            "Liquidez Seca": ls.iloc[i],
            "Liquidez Imediata": li.iloc[i],
            "Liquidez Geral": lg.iloc[i],
            "PCT": pct.iloc[i],
            "CE": ce.iloc[i],
            "IPL": ipl.iloc[i]
        })

kpis_final = pd.DataFrame(registros)

print("Dimensões da tabela final:", kpis_final.shape)
display(kpis_final.head(12))

## 22. Validação dos indicadores

A tabela abaixo apresenta uma visão resumida da validação da base e dos indicadores calculados.



In [49]:
# Validação estrutural da tabela final

validacao = pd.DataFrame({
    "Indicador": [
        "Receita",
        "Custos",
        "EBITDA",
        "Margem EBITDA",
        "Resultado Líquido",
        "Margem Líquida",
        "Liquidez Corrente",
        "Liquidez Seca",
        "Liquidez Imediata",
        "Liquidez Geral",
        "PCT",
        "CE",
        "IPL"
    ],
    "Status": [
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK",
        "OK*",
        "OK",
        "OK*",
        "OK*",
        "OK*",
        "OK*"
    ],
    "Observação": [
        "Disponível na DRE",
        "Disponível como custo total",
        "Fornecido pela base",
        "Calculada sobre a receita",
        "Disponível na DRE",
        "Calculada sobre a receita",
        "Calculada com dados do BP",
        "Ano 12 possui ausência estrutural de estoque",
        "Calculada com o disponível",
        "Ano 12 possui ausência estrutural no ELP",
        "Calculado com capital de terceiros",
        "Calculado com passivo circulante",
        "Calculado com permanente e PL"
    ]
})

display(validacao)

,Indicador,Status,Observação
0,Receita,OK,Disponível na DRE
1,Custos,OK,Disponível como custo total
2,EBITDA,OK,Fornecido pela base
3,Margem EBITDA,OK,Calculada sobre a receita
4,Resultado Líquido,OK,Disponível na DRE
5,Margem Líquida,OK,Calculada sobre a receita
6,Liquidez Corrente,OK,Calculada com dados do BP
7,Liquidez Seca,OK*,Ano 12 possui ausência estrutural de estoque
8,Liquidez Imediata,OK,Calculada com o disponível
9,Liquidez Geral,OK*,Ano 12 possui ausência estrutural no ELP


In [50]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter

# ============================================================
# 1. Carregar a base tratada
# ============================================================

df = pd.read_csv(
    "../staging/dados_tratados.csv",
    sep=";",
    encoding="utf-8-sig"
)

cenario = "Total Cen_00001"
anos = [f"Ano {i}" for i in range(1, 13)]

dados = (
    df[df["Cenario"] == cenario]
    .pivot_table(
        index="Conta",
        columns="Ano",
        values="Valor_numerico",
        aggfunc="first"
    )
    .reindex(columns=anos)
)

# ============================================================
# 2. Criar o arquivo Excel
# ============================================================

wb = Workbook()
ws = wb.active
ws.title = "Consulta_Cenario"

# ============================================================
# 3. Estilos
# ============================================================

azul = "5B9BD5"
azul_escuro = "1F4E78"
azul_claro = "D9EAF7"
verde = "E2F0D9"
cinza = "D9E1F2"
branco = "FFFFFF"

borda = Border(
    bottom=Side(style="thin", color="B7B7B7")
)

# ============================================================
# 4. Título
# ============================================================

ws.merge_cells("A1:M1")
ws["A1"] = "Demonstrativos Contábeis – Anos 1 a 12 – Consulta por Cenário"

ws["A1"].font = Font(
    bold=True,
    size=15,
    color=branco
)

ws["A1"].fill = PatternFill(
    "solid",
    fgColor=azul_escuro
)

ws["A1"].alignment = Alignment(
    horizontal="left",
    vertical="center"
)

ws.row_dimensions[1].height = 25

# ============================================================
# 5. Seleção do cenário
# ============================================================

ws["A3"] = "Selecione o cenário:"
ws["A3"].font = Font(bold=True)

ws["B3"] = cenario
ws["B3"].font = Font(bold=True)
ws["B3"].fill = PatternFill("solid", fgColor=verde)
ws["B3"].alignment = Alignment(horizontal="center")

ws["D3"] = "Esta consulta utiliza 1 dos 1.200 cenários disponíveis."
ws["D3"].font = Font(
    italic=True,
    color="666666"
)

# ============================================================
# 6. Cabeçalho de anos
# ============================================================

def criar_cabecalho(linha):
    ws.cell(linha, 1, "Conta")

    for coluna, ano in enumerate(anos, start=2):
        ws.cell(linha, coluna, ano)

    for coluna in range(1, 14):
        celula = ws.cell(linha, coluna)

        celula.font = Font(
            bold=True,
            color=branco
        )

        celula.fill = PatternFill(
            "solid",
            fgColor=azul_escuro
        )

        celula.alignment = Alignment(
            horizontal="center"
        )

        celula.border = borda


# ============================================================
# 7. Formatação dos valores
# ============================================================

def colocar_valor(celula, valor):
    if pd.isna(valor):
        celula.value = None
    else:
        # O modelo do professor trabalha com valores em R$ mil
        celula.value = valor / 1000

        # Positivo: 1.234.567
        # Negativo: (1.234.567)
        celula.number_format = '#,##0;(#,##0);-'

    celula.alignment = Alignment(
        horizontal="right"
    )


def escrever_conta(linha, nome, valores, negrito=False):

    ws.cell(linha, 1, nome)

    ws.cell(linha, 1).font = Font(
        bold=negrito
    )

    for coluna, ano in enumerate(anos, start=2):
        colocar_valor(
            ws.cell(linha, coluna),
            valores.get(ano)
        )

    for coluna in range(1, 14):
        ws.cell(linha, coluna).border = borda


# ============================================================
# 8. Criar seção
# ============================================================

def criar_secao(linha, titulo):

    ws.merge_cells(
        start_row=linha,
        start_column=1,
        end_row=linha,
        end_column=13
    )

    celula = ws.cell(linha, 1, titulo)

    celula.font = Font(
        bold=True,
        color=branco
    )

    celula.fill = PatternFill(
        "solid",
        fgColor=azul
    )

    celula.alignment = Alignment(
        horizontal="center"
    )

    return linha + 1


# ============================================================
# 9. BALANÇO PATRIMONIAL
# ============================================================

linha = 5

linha = criar_secao(
    linha,
    "BALANÇO PATRIMONIAL"
)

criar_cabecalho(linha)
linha += 1

ordem_bal = [
    "BAL - Ativo Circulante",
    "BAL - Disponível",
    "BAL - Contas a Receber - Clientes",
    "BAL - Contas a Receber - Partes Relacionadas",
    "BAL - Contas a Receber - SWAP",
    "BAL - Estoques Diversos",
    "BAL - Créditos Tributários",
    "BAL - Outros Créditos",
    "BAL - Realizável a Longo Prazo",
    "BAL - Outros Créditos LP",
    "BAL - At Fiscal Diferido",
    "BAL - Investimentos - Imobilizado",
    "BAL - Investimentos - Intangível",
    "BAL - Outorga da Concessão",
    "BAL - Diferido",
    "BAL - Amortização - Intangível",
    "BAL - Amortização Acumulada",
    "BAL - Depreciação Acumulada",
    "BAL - Permanente",
    "BAL - Total do Ativo",
    "BAL - Passivo Circulante",
    "BAL - Fornecedores",
    "BAL - Contas a Pagar - Parte Relacionada",
    "BAL - Emprést",
    "BAL - Empréstimos",
    "BAL - Tributos a pagar",
    "BAL - Impostos",
    "BAL - Encargos Sociais e Trabalhistas",
    "BAL - Obrigações com o Poder Concedente",
    "BAL - Provisão Manutenção",
    "BAL - Prov para Contingências",
    "BAL - Outros Débitos",
    "BAL - Outros deb",
    "BAL - Exigível a Longo Prazo",
    "BAL - Total do Passivo",
    "BAL - Patrimônio Líquido",
    "BAL - Capital Social",
    "BAL - Reservas Legais",
    "BAL - Reservas de Capital",
    "BAL - Reserva de Retenção de Lucros",
    "BAL - Resultado Acumulado",
    "BAL - Resultado do Período",
    "BAL - Dividendos Antecipados"
]

for conta in ordem_bal:
    if conta in dados.index:
        escrever_conta(
            linha,
            conta.replace("BAL - ", ""),
            dados.loc[conta],
            negrito=(
                "Total" in conta
                or "Patrimônio Líquido" in conta
                or conta == "BAL - Ativo Circulante"
                or conta == "BAL - Passivo Circulante"
            )
        )
        linha += 1

linha += 1


# ============================================================
# 10. DRE — formato do professor
# ============================================================

linha = criar_secao(
    linha,
    "DEMONSTRAÇÃO DO RESULTADO (DRE)"
)

criar_cabecalho(linha)
linha += 1

receita = dados.loc["DRE - Receita"]
tributos = dados.loc["DRE - Tributos"]
custos = dados.loc["DRE - Custos"]
da = dados.loc["DRE - Depreciação e Amortização"]
resultado_operacional = dados.loc["DRE - Resultado Operacional"]
receitas_financeiras = dados.loc["DRE - Receitas Financeiras"]
despesas_financeiras = dados.loc["DRE - Despesas Financeiras"]
resultado_financeiro = dados.loc["DRE - Resultado Financeiro"]
outros_resultados = dados.loc["DRE - Outros Resultados Operacionais"]
resultado_antes_ir = dados.loc[
    "DRE - Resultado Antes do Imposto de Renda"
]
ir_cs = dados.loc[
    "DRE - Imposto de Renda e Contribuição Social"
]
resultado_liquido = dados.loc[
    "DRE - Resultado Líquido"
]
resultado_equivalencia = dados.loc[
    "DRE - Resultado Líquido após Equivalência"
]
ebitda = dados.loc["DRE - EBITDA"]

# Receita Operacional
escrever_conta(
    linha,
    "Receita Operacional",
    receita,
    negrito=True
)
linha += 1

# Tributos
escrever_conta(
    linha,
    "(-) Tributos sobre a Receita",
    tributos
)
linha += 1

# Receita Líquida = Receita + Tributos
receita_liquida = receita + tributos

escrever_conta(
    linha,
    "Receita Líquida",
    receita_liquida,
    negrito=True
)
linha += 1

# Custos
escrever_conta(
    linha,
    "(-) Custos",
    custos
)
linha += 1

# Depreciação
escrever_conta(
    linha,
    "(-) Depreciação e Amortização",
    da
)
linha += 1

# Resultado Operacional
escrever_conta(
    linha,
    "Resultado Operacional",
    resultado_operacional,
    negrito=True
)
linha += 1

# Receitas Financeiras
escrever_conta(
    linha,
    "Receitas Financeiras",
    receitas_financeiras
)
linha += 1

# Despesas Financeiras
escrever_conta(
    linha,
    "(-) Despesas Financeiras",
    despesas_financeiras
)
linha += 1

# Resultado Financeiro
escrever_conta(
    linha,
    "Resultado Financeiro",
    resultado_financeiro,
    negrito=True
)
linha += 1

# Outros resultados
escrever_conta(
    linha,
    "Outros Resultados Operacionais",
    outros_resultados
)
linha += 1

# Resultado antes IR
escrever_conta(
    linha,
    "Resultado Antes do IR e CS",
    resultado_antes_ir,
    negrito=True
)
linha += 1

# IR
escrever_conta(
    linha,
    "(-) Imposto de Renda e Contribuição Social",
    ir_cs
)
linha += 1

# Resultado líquido
escrever_conta(
    linha,
    "RESULTADO LÍQUIDO",
    resultado_liquido,
    negrito=True
)
linha += 1

# Equivalência
escrever_conta(
    linha,
    "Resultado Líquido após Equivalência",
    resultado_equivalencia
)
linha += 2


# ============================================================
# 11. INDICADORES
# ============================================================

ws.merge_cells(
    start_row=linha,
    start_column=1,
    end_row=linha,
    end_column=13
)

ws.cell(
    linha,
    1,
    "INDICADORES (calculados)"
)

ws.cell(linha, 1).font = Font(
    bold=True,
    color=branco
)

ws.cell(linha, 1).fill = PatternFill(
    "solid",
    fgColor=azul
)

ws.cell(linha, 1).alignment = Alignment(
    horizontal="center"
)

linha += 1

# EBITDA
escrever_conta(
    linha,
    "EBITDA (consensus model)",
    ebitda,
    negrito=True
)
linha += 1

# Margem EBITDA
margem_ebitda = ebitda / receita_liquida

ws.cell(linha, 1, "Margem EBITDA (s/ Receita Líquida)")
ws.cell(linha, 1).font = Font(bold=True)

for coluna, ano in enumerate(anos, start=2):
    celula = ws.cell(linha, coluna)

    valor = margem_ebitda.get(ano)

    if pd.isna(valor):
        celula.value = None
    else:
        celula.value = valor
        celula.number_format = "0.0%"

    celula.alignment = Alignment(
        horizontal="right"
    )

linha += 1

# Margem líquida
margem_liquida = resultado_liquido / receita_liquida

ws.cell(
    linha,
    1,
    "Margem Líquida (s/ Receita Líquida)"
)

ws.cell(linha, 1).font = Font(bold=True)

for coluna, ano in enumerate(anos, start=2):
    celula = ws.cell(linha, coluna)

    valor = margem_liquida.get(ano)

    if pd.isna(valor):
        celula.value = None
    else:
        celula.value = valor
        celula.number_format = "0.0%"

    celula.alignment = Alignment(
        horizontal="right"
    )

linha += 2


# ============================================================
# 12. Fluxo de Caixa / DFC
# ============================================================

linha = criar_secao(
    linha,
    "DEMONSTRAÇÃO DOS FLUXOS DE CAIXA (DFC)"
)

criar_cabecalho(linha)
linha += 1

ordem_flu = [
    "FLU - Entradas",
    "FLU - Receita",
    "FLU - Tributos",
    "FLU - Custos",
    "FLU - Resultado Financeiro",
    "FLU - Despesas Financeiras",
    "FLU - Imposto de Renda e Contribuição Social",
    "FLU - Geração de Caixa",
    "FLU - Investimentos",
    "FLU - Distribuição para Acionista",
    "FLU - Saldo Inicial",
    "FLU - Saldo Final"
]

for conta in ordem_flu:
    if conta in dados.index:
        escrever_conta(
            linha,
            conta.replace("FLU - ", ""),
            dados.loc[conta],
            negrito=(
                conta in [
                    "FLU - Geração de Caixa",
                    "FLU - Saldo Final"
                ]
            )
        )
        linha += 1


# ============================================================
# 13. Pendências
# ============================================================

linha += 2

ws.merge_cells(
    start_row=linha,
    start_column=1,
    end_row=linha,
    end_column=13
)

ws.cell(
    linha,
    1,
    "PENDÊNCIAS DE DEFINIÇÃO"
)

ws.cell(linha, 1).font = Font(
    bold=True,
    color=branco
)

ws.cell(linha, 1).fill = PatternFill(
    "solid",
    fgColor=azul_escuro
)

linha += 1

pendencias = [
    "Custos variáveis — depende de metodologia/premissa",
    "CAC / LTV — depende da definição da simulação",
    "EVA — faltam WACC e capital investido confiável",
    "MVA — falta valor de mercado"
]

for pendencia in pendencias:
    ws.cell(linha, 1, pendencia)
    ws.cell(linha, 1).fill = PatternFill(
        "solid",
        fgColor="FFF2CC"
    )
    linha += 1


# ============================================================
# 14. Ajustes visuais finais
# ============================================================

ws.column_dimensions["A"].width = 48

for coluna in range(2, 14):
    ws.column_dimensions[
        get_column_letter(coluna)
    ].width = 15

ws.freeze_panes = "B7"

ws.sheet_view.showGridLines = False

# Alinhamento geral
for row in ws.iter_rows():
    for celula in row:
        if celula.value is not None:
            celula.alignment = Alignment(
                vertical="center",
                horizontal=(
                    "left"
                    if celula.column == 1
                    else "right"
                )
            )

# Salvar
arquivo = "../staging/indicadores.xlsx"

wb.save(arquivo)

print("Planilha criada com sucesso!")
print("Arquivo:", arquivo)
print("Cenário:", cenario)
print("Estrutura: BP + DRE + Indicadores + DFC")

Planilha criada com sucesso!
Arquivo: ../staging/indicadores.xlsx
Cenário: Total Cen_00001
Estrutura: BP + DRE + Indicadores + DFC


## Índices de Endividamento — Cenário 1

Os índices de endividamento permitem analisar a participação do capital de terceiros na estrutura financeira da empresa.

Para o cenário selecionado, serão calculados:

- Participação de Capital de Terceiros (PCT);
- Composição do Endividamento (CE);
- Imobilização do Patrimônio Líquido (IPL).

Os cálculos serão realizados para os 12 anos disponíveis na base.

In [51]:
# Valores do balanço utilizados nos cálculos
passivo_circulante = bal_organizado.loc[
    "BAL - Passivo Circulante"
].abs()

exigivel_lp = bal_organizado.loc[
    "BAL - Exigível a Longo Prazo"
].abs()

patrimonio_liquido = bal_organizado.loc[
    "BAL - Patrimônio Líquido"
].abs()

permanente = bal_organizado.loc[
    "BAL - Permanente"
]

# Capital de terceiros = dívidas de curto + longo prazo
capital_terceiros = passivo_circulante + exigivel_lp

# Índices de endividamento
pct = capital_terceiros / patrimonio_liquido

ce = passivo_circulante / capital_terceiros

ipl = permanente / patrimonio_liquido

# Tabela final
indices_endividamento = pd.DataFrame({
    "Passivo Circulante": passivo_circulante,
    "Exigível a Longo Prazo": exigivel_lp,
    "Patrimônio Líquido": patrimonio_liquido,
    "Capital de Terceiros": capital_terceiros,
    "PCT": pct,
    "CE": ce,
    "IPL": ipl
})

display(indices_endividamento)

,Passivo Circulante,Exigível a Longo Prazo,Patrimônio Líquido,Capital de Terceiros,PCT,CE,IPL
Ano,,,,,,,
Ano 1,1.080928e+09,7.535764e+09,2.014435e+09,8.616692e+09,4.277473,0.125446,4.068824
Ano 2,1.357705e+09,7.535764e+09,1.461523e+09,8.893468e+09,6.085068,0.152663,5.391876
Ano 3,1.762031e+09,7.535764e+09,1.514931e+09,9.297795e+09,6.137439,0.189511,4.826759
Ano 4,4.980976e+08,7.535764e+09,1.612422e+09,8.033861e+09,4.982482,0.062000,4.104556
Ano 5,1.263215e+09,7.535764e+09,1.794732e+09,8.798979e+09,4.902671,0.143564,3.266220
Ano 6,2.001197e+09,7.535764e+09,1.989532e+09,9.536960e+09,4.793570,0.209836,2.577095
Ano 7,2.475828e+09,7.535764e+09,2.152697e+09,1.001159e+10,4.650721,0.247296,2.010339
Ano 8,2.327826e+09,7.531151e+09,2.380147e+09,9.858978e+09,4.142171,0.236112,1.465344
Ano 9,2.729359e+09,7.526397e+09,2.663144e+09,1.025576e+10,3.850995,0.266129,1.036045


In [52]:
# Componentes disponíveis para análise do EVA

nopat_conta = "DRE - Resultado Operacional"
imposto_conta = "DRE - Imposto de Renda e Contribuição Social"

contas_capital = [
    "BAL - Ativo Circulante",
    "BAL - Realizável a Longo Prazo",
    "BAL - Permanente",
    "BAL - Passivo Circulante",
    "BAL - Exigível a Longo Prazo",
    "BAL - Patrimônio Líquido"
]

# Verifica quais contas existem na base organizada
componentes_eva = pd.DataFrame(index=ordem_anos)

componentes_eva["Resultado Operacional"] = dre_organizada.loc[
    nopat_conta
].values

componentes_eva["IR e CS"] = dre_organizada.loc[
    imposto_conta
].values

for conta in contas_capital:
    componentes_eva[conta.replace("BAL - ", "")] = bal_organizado.loc[
        conta
    ].values

display(componentes_eva)

,Resultado Operacional,IR e CS,Ativo Circulante,Realizável a Longo Prazo,Permanente,Passivo Circulante,Exigível a Longo Prazo,Patrimônio Líquido
Ano 1,4.286529e+09,-1.280022e+09,1.795721e+09,6.390239e+08,8.196382e+09,-1.080928e+09,-7.535764e+09,-2.014435e+09
Ano 2,3.549559e+09,-1.054053e+09,1.567979e+09,9.066612e+08,7.880352e+09,-1.357705e+09,-7.535764e+09,-1.461523e+09
Ano 3,3.709887e+09,-1.119699e+09,2.292584e+09,1.207936e+09,7.312205e+09,-1.762031e+09,-7.535764e+09,-1.514931e+09
Ano 4,3.899047e+09,-1.219618e+09,1.655697e+09,1.372312e+09,6.618274e+09,-4.980976e+08,-7.535764e+09,-1.612422e+09
Ano 5,4.298228e+09,-1.387012e+09,8.393786e+08,1.365914e+09,5.861988e+09,1.263215e+09,-7.535764e+09,-1.794732e+09
Ano 6,4.657110e+09,-1.549159e+09,7.907685e+08,1.606117e+09,5.127214e+09,2.001197e+09,-7.535764e+09,-1.989532e+09
Ano 7,4.913431e+09,-1.652172e+09,9.407401e+08,1.944243e+09,4.327649e+09,2.475828e+09,-7.535764e+09,-2.152697e+09
Ano 8,5.303891e+09,-1.794810e+09,1.779399e+09,2.316339e+09,3.487735e+09,2.327826e+09,-7.531151e+09,-2.380147e+09
Ano 9,5.683662e+09,-1.953967e+09,2.208646e+09,2.492400e+09,2.759136e+09,2.729359e+09,-7.526397e+09,-2.663144e+09
Ano 10,6.319144e+09,-2.214465e+09,2.461672e+09,2.431772e+09,1.923826e+09,3.806934e+09,-7.521431e+09,-3.102773e+09


In [53]:
# Resultado operacional utilizado como referência para o EBIT
ebit = dre_organizada.loc[
    "DRE - Resultado Operacional"
]

# Resultado antes do IR e CS
resultado_antes_ir = dre_organizada.loc[
    "DRE - Resultado Antes do Imposto de Renda"
]

# Imposto de Renda e Contribuição Social
ir_cs = dre_organizada.loc[
    "DRE - Imposto de Renda e Contribuição Social"
]

# Alíquota efetiva de IR e CS
aliquota_ir_cs = ir_cs.abs() / resultado_antes_ir

# Imposto estimado sobre o resultado operacional
imposto_sobre_ebit = ebit * aliquota_ir_cs

# NOPAT
nopat = ebit - imposto_sobre_ebit

nopat_df = pd.DataFrame({
    "EBIT / Resultado Operacional": ebit,
    "Resultado Antes do IR e CS": resultado_antes_ir,
    "IR e CS": ir_cs,
    "Alíquota Efetiva": aliquota_ir_cs,
    "Imposto sobre EBIT": imposto_sobre_ebit,
    "NOPAT": nopat
})

display(nopat_df)

,EBIT / Resultado Operacional,Resultado Antes do IR e CS,IR e CS,Alíquota Efetiva,Imposto sobre EBIT,NOPAT
Ano,,,,,,
Ano 1,4.286529e+09,3.723265e+09,-1.280022e+09,0.343790,1.473667e+09,2.812862e+09
Ano 2,3.549559e+09,3.054887e+09,-1.054053e+09,0.345038,1.224734e+09,2.324825e+09
Ano 3,3.709887e+09,3.242845e+09,-1.119699e+09,0.345283,1.280961e+09,2.428926e+09
Ano 4,3.899047e+09,3.537745e+09,-1.219618e+09,0.344744,1.344174e+09,2.554873e+09
Ano 5,4.298228e+09,4.029618e+09,-1.387012e+09,0.344204,1.479469e+09,2.818760e+09
Ano 6,4.657110e+09,4.504528e+09,-1.549159e+09,0.343911,1.601633e+09,3.055476e+09
Ano 7,4.913431e+09,4.805479e+09,-1.652172e+09,0.343810,1.689287e+09,3.224144e+09
Ano 8,5.303891e+09,5.222846e+09,-1.794810e+09,0.343646,1.822661e+09,3.481231e+09
Ano 9,5.683662e+09,5.688727e+09,-1.953967e+09,0.343481,1.952228e+09,3.731434e+09


In [56]:
# Capital Investido conforme a metodologia apresentada pelo professor

patrimonio_liquido = bal_organizado.loc[
    "BAL - Patrimônio Líquido"
].abs()

emprestimo = bal_organizado.loc[
    "BAL - Emprést"
].abs()

emprestimos = bal_organizado.loc[
    "BAL - Empréstimos"
].abs()

divida_financeira = emprestimo + emprestimos

capital_investido = (
    patrimonio_liquido
    + divida_financeira
)

capital_investido_df = pd.DataFrame({
    "Patrimônio Líquido": patrimonio_liquido,
    "Emprést": emprestimo,
    "Empréstimos": emprestimos,
    "Dívida Financeira": divida_financeira,
    "Capital Investido": capital_investido
})

display(capital_investido_df)

,Patrimônio Líquido,Emprést,Empréstimos,Dívida Financeira,Capital Investido
Ano,,,,,
Ano 1,2.014435e+09,5.649257e+09,2.791024e+08,5.928359e+09,7.942794e+09
Ano 2,1.461523e+09,5.649257e+09,3.283839e+08,5.977641e+09,7.439164e+09
Ano 3,1.514931e+09,5.649257e+09,3.692838e+08,6.018540e+09,7.533471e+09
Ano 4,1.612422e+09,5.649257e+09,8.387912e+08,6.488048e+09,8.100469e+09
Ano 5,1.794732e+09,5.649257e+09,2.040214e+09,7.689471e+09,9.484202e+09
Ano 6,1.989532e+09,5.649257e+09,2.898691e+09,8.547948e+09,1.053748e+10
Ano 7,2.152697e+09,5.649257e+09,3.757705e+09,9.406962e+09,1.155966e+10
Ano 8,2.380147e+09,5.644644e+09,4.077711e+09,9.722355e+09,1.210250e+10
Ano 9,2.663144e+09,5.639890e+09,4.418330e+09,1.005822e+10,1.272136e+10


In [55]:
# Despesas financeiras
despesas_financeiras = dre_organizada.loc[
    "DRE - Despesas Financeiras"
].abs()

# Custo aproximado da dívida financeira
custo_divida = despesas_financeiras / divida_financeira

analise_wacc = pd.DataFrame({
    "Dívida Financeira": divida_financeira,
    "Despesas Financeiras": despesas_financeiras,
    "Custo Aproximado da Dívida": custo_divida
})

display(analise_wacc)

,Dívida Financeira,Despesas Financeiras,Custo Aproximado da Dívida
Ano,,,
Ano 1,5.928359e+09,6.939747e+08,0.117060
Ano 2,5.977641e+09,6.888306e+08,0.115235
Ano 3,6.018540e+09,6.240541e+08,0.103689
Ano 4,6.488048e+09,5.940416e+08,0.091559
Ano 5,7.689471e+09,4.324440e+08,0.056238
Ano 6,8.547948e+09,2.316342e+08,0.027098
Ano 7,9.406962e+09,1.841490e+08,0.019576
Ano 8,9.722355e+09,1.754314e+08,0.018044
Ano 9,1.005822e+10,1.884766e+08,0.018739


In [57]:
roic = nopat / capital_investido

roic_df = pd.DataFrame({
    "NOPAT": nopat,
    "Capital Investido": capital_investido,
    "ROIC": roic
})

display(roic_df)

,NOPAT,Capital Investido,ROIC
Ano,,,
Ano 1,2.812862e+09,7.942794e+09,0.354140
Ano 2,2.324825e+09,7.439164e+09,0.312512
Ano 3,2.428926e+09,7.533471e+09,0.322418
Ano 4,2.554873e+09,8.100469e+09,0.315398
Ano 5,2.818760e+09,9.484202e+09,0.297206
Ano 6,3.055476e+09,1.053748e+10,0.289963
Ano 7,3.224144e+09,1.155966e+10,0.278913
Ano 8,3.481231e+09,1.210250e+10,0.287646
Ano 9,3.731434e+09,1.272136e+10,0.293320


In [58]:
disponivel = bal_organizado.loc[
    "BAL - Disponível"
].abs()

capital_investido_eva = (
    patrimonio_liquido
    + emprestimo
    + emprestimos
    - disponivel
)

wacc = 0.08

encargo_capital = capital_investido_eva * wacc

eva = nopat - encargo_capital

eva_df = pd.DataFrame({
    "NOPAT": nopat,
    "Capital Investido": capital_investido_eva,
    "WACC": wacc,
    "Encargo de Capital": encargo_capital,
    "EVA": eva
})

display(eva_df)

,NOPAT,Capital Investido,WACC,Encargo de Capital,EVA
Ano,,,,,
Ano 1,2.812862e+09,6.737529e+09,0.08,5.390023e+08,2.273859e+09
Ano 2,2.324825e+09,6.416736e+09,0.08,5.133389e+08,1.811486e+09
Ano 3,2.428926e+09,5.785050e+09,0.08,4.628040e+08,1.966122e+09
Ano 4,2.554873e+09,6.983079e+09,0.08,5.586463e+08,1.996226e+09
Ano 5,2.818760e+09,9.176816e+09,0.08,7.341453e+08,2.084614e+09
Ano 6,3.055476e+09,1.027130e+10,0.08,8.217044e+08,2.233772e+09
Ano 7,3.224144e+09,1.113197e+10,0.08,8.905573e+08,2.333587e+09
Ano 8,3.481231e+09,1.073254e+10,0.08,8.586030e+08,2.622628e+09
Ano 9,3.731434e+09,1.079803e+10,0.08,8.638425e+08,2.867592e+09


In [59]:
mva_aproximado = eva / wacc

mva_df = pd.DataFrame({
    "EVA": eva,
    "WACC": wacc,
    "MVA Aproximado": mva_aproximado
})

display(mva_df)

,EVA,WACC,MVA Aproximado
Ano,,,
Ano 1,2.273859e+09,0.08,2.842324e+10
Ano 2,1.811486e+09,0.08,2.264357e+10
Ano 3,1.966122e+09,0.08,2.457653e+10
Ano 4,1.996226e+09,0.08,2.495283e+10
Ano 5,2.084614e+09,0.08,2.605768e+10
Ano 6,2.233772e+09,0.08,2.792215e+10
Ano 7,2.333587e+09,0.08,2.916983e+10
Ano 8,2.622628e+09,0.08,3.278285e+10
Ano 9,2.867592e+09,0.08,3.584490e+10
